# 03 — Train/Test Split et Preprocessing Machine Learning

## Objectif

Ce notebook prépare le dataset issu du Feature Engineering pour l'entraînement
des modèles de régression.

Les objectifs sont :

- charger le dataset de features ;
- séparer les variables explicatives `X` et la cible `y` ;
- créer les jeux d'entraînement et de test ;
- identifier les variables numériques et catégorielles ;
- définir les stratégies d'imputation ;
- définir les stratégies d'encodage des variables catégorielles ;
- construire un pipeline de preprocessing `scikit-learn` ;
- ajuster le preprocessing uniquement sur le jeu d'entraînement ;
- transformer les jeux d'entraînement et de test ;
- vérifier la cohérence des matrices obtenues.

La variable cible est :

`co2_wltp_g_km`

Aucune transformation dépendant des données n'est ajustée avant la séparation
train / test.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "03_ml_preprocessing":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

INPUT_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "data_2024_features.csv"
)

TARGET = "co2_wltp_g_km"

TEST_MODE = True
NROWS_TEST = 100_000

TEST_SIZE = 0.20
RANDOM_STATE = 42

## 1. Chargement du dataset de features

### Objectif

Cette étape charge le dataset produit par le pipeline de Feature Engineering.

Deux modes sont utilisés :

- **Mode TEST** : 100 000 observations pour le développement local ;
- **Mode COMPLET** : intégralité du dataset pour la validation finale.

Aucune transformation n'est appliquée lors du chargement.

In [2]:
if not INPUT_DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset de features introuvable : {INPUT_DATA_PATH}"
    )

if TEST_MODE:
    df = pd.read_csv(
        INPUT_DATA_PATH,
        nrows=NROWS_TEST,
        low_memory=False,
    )

    print(
        f"Mode TEST : {len(df):,} observations chargées."
    )
else:
    df = pd.read_csv(
        INPUT_DATA_PATH,
        low_memory=False,
    )

    print(
        f"Mode COMPLET : {len(df):,} observations chargées."
    )

print(f"Shape : {df.shape}")

Mode TEST : 100,000 observations chargées.
Shape : (100000, 15)


## 2. Contrôle du schéma d'entrée

### Objectif

Avant la séparation des données, cette étape vérifie :

- la présence de la variable cible ;
- l'absence de valeurs manquantes dans la cible ;
- l'unicité des noms de colonnes ;
- la cohérence générale du dataset de features.

In [3]:
if TARGET not in df.columns:
    raise ValueError(
        f"Variable cible absente : {TARGET}"
    )

if df[TARGET].isna().any():
    raise ValueError(
        "La variable cible contient des valeurs manquantes."
    )

if not df.columns.is_unique:
    raise ValueError(
        "Les noms de colonnes ne sont pas uniques."
    )

print("✅ Schéma d'entrée valide.")

✅ Schéma d'entrée valide.


## 3. Séparation entre variables explicatives et cible

### Objectif

Cette étape construit :

- `X` : ensemble des variables explicatives ;
- `y` : variable cible `co2_wltp_g_km`.

La cible est retirée de `X` avant toute opération de preprocessing.

In [4]:
X = df.drop(
    columns=[TARGET]
).copy()

y = df[TARGET].copy()

print(f"X : {X.shape}")
print(f"y : {y.shape}")

X : (100000, 14)
y : (100000,)


## 4. Séparation Train / Test

### Objectif

Le dataset est séparé avant toute imputation, encodage ou standardisation.

Cette séparation garantit que les paramètres du preprocessing sont appris
uniquement à partir du jeu d'entraînement.

La répartition retenue est :

- 80 % pour l'entraînement ;
- 20 % pour le test ;
- `random_state = 42` pour assurer la reproductibilité.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(
    f"X_train : {X_train.shape}"
)
print(
    f"X_test  : {X_test.shape}"
)
print(
    f"y_train : {y_train.shape}"
)
print(
    f"y_test  : {y_test.shape}"
)

X_train : (80000, 14)
X_test  : (20000, 14)
y_train : (80000,)
y_test  : (20000,)


## 5. Identification des variables numériques et catégorielles

### Objectif

Le preprocessing dépend du type de variable.

Les variables sont séparées en :

- variables numériques ;
- variables catégorielles.

Cette distinction permettra de construire des pipelines de transformation
adaptés à chaque groupe.

In [6]:
numeric_columns = (
    X_train
    .select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)

categorical_columns = (
    X_train
    .select_dtypes(
        include=["object", "string", "category"]
    )
    .columns
    .tolist()
)

print(
    f"Variables numériques     : {len(numeric_columns)}"
)
print(
    f"Variables catégorielles : {len(categorical_columns)}"
)

Variables numériques     : 10
Variables catégorielles : 4


## 6. Analyse des valeurs manquantes après séparation Train / Test

### Objectif

Cette étape analyse les valeurs manquantes dans le jeu d'entraînement afin de
définir les stratégies d'imputation adaptées.

Les statistiques sont calculées uniquement sur `X_train`.

In [7]:
missing_summary = pd.DataFrame({
    "column": X_train.columns,
    "dtype": X_train.dtypes.astype(str).values,
    "missing_count": X_train.isna().sum().values,
    "missing_pct": (
        X_train.isna().mean().values * 100
    ),
})

missing_summary = (
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
    .sort_values(
        "missing_pct",
        ascending=False,
    )
)

display(missing_summary)

,column,dtype,missing_count,missing_pct
11,electric_range_km,float64,63215,79.01875
8,electric_energy_consumption_wh_km,float64,63196,78.99500
9,co2_reduction_wltp_g_km,float64,35042,43.80250
10,fuel_consumption,float64,11677,14.59625
6,engine_capacity_cm3,float64,11355,14.19375
3,wltp_test_mass_kg,float64,225,0.28125
0,manufacturer_make,object,3,0.00375
7,engine_power_kw,float64,1,0.00125


## 7. Analyse de la cardinalité des variables catégorielles du Train

### Objectif

Cette étape mesure la cardinalité réelle des variables catégorielles dans le
jeu d'entraînement.

Cette information servira à choisir les stratégies d'encodage sans utiliser
les données du jeu de test.

In [8]:
categorical_cardinality = pd.DataFrame({
    "column": categorical_columns,
    "unique_values": [
        X_train[column].nunique(
            dropna=False
        )
        for column in categorical_columns
    ],
    "missing_pct": [
        X_train[column].isna().mean() * 100
        for column in categorical_columns
    ],
})

categorical_cardinality = (
    categorical_cardinality
    .sort_values(
        "unique_values",
        ascending=True,
    )
)

display(categorical_cardinality)

,column,unique_values,missing_pct
1,vehicle_category_type,2,0.00000
3,fuel_mode,6,0.00000
2,fuel_type,9,0.00000
0,manufacturer_make,94,0.00375


## 8. Définition des groupes de variables pour le preprocessing

### Objectif

Les variables sont regroupées selon leur type afin de définir les
transformations adaptées au pipeline de preprocessing.

Le dataset contient :

- des variables numériques ;
- quatre variables catégorielles nominales de cardinalité maîtrisée.

Les variables numériques seront traitées par une stratégie d'imputation
adaptée.

Les variables catégorielles seront imputées si nécessaire puis encodées par
One-Hot Encoding.

Les transformations seront ajustées uniquement à partir du jeu
d'entraînement.

In [9]:
categorical_columns_for_encoding = (
    categorical_columns.copy()
)

print(
    f"Variables numériques : "
    f"{len(numeric_columns)}"
)

print(
    f"Variables catégorielles : "
    f"{len(categorical_columns_for_encoding)}"
)

print("\nVariables catégorielles à encoder :")

for column in categorical_columns_for_encoding:
    print(f"  - {column}")

Variables numériques : 10
Variables catégorielles : 4

Variables catégorielles à encoder :
  - manufacturer_make
  - vehicle_category_type
  - fuel_type
  - fuel_mode


## 9. Analyse du caractère nominal ou ordinal des variables catégorielles

### Objectif

Avant de définir l'encodage, cette étape examine les modalités des variables
catégorielles conservées afin de déterminer si elles possèdent ou non un ordre
métier naturel.

Une variable est considérée comme :

- **nominale** lorsque ses modalités représentent des catégories sans ordre
  intrinsèque ;
- **ordinale** lorsque ses modalités représentent des niveaux pouvant être
  classés selon un ordre métier explicite.

L'inspection des modalités permet de prendre cette décision avant la
construction du pipeline d'encodage.

In [10]:
for column in categorical_columns:
    values = (
        X_train[column]
        .dropna()
        .astype(str)
        .unique()
    )

    values = sorted(values)

    print(
        f"\n{column} "
        f"({X_train[column].nunique(dropna=False):,} modalités)"
    )

    print(values[:30])

    if len(values) > 30:
        print("...")


manufacturer_make (94 modalités)
['ABARTH', 'AIWAYS', 'ALFA ROMEO', 'ALPINA', 'ALPINE', 'ASTON MARTIN', 'AUDI', 'AUDI AG', 'BAIC', 'BENTLEY', 'BMW', 'BMW BRILLIANCE', 'BRABUS', 'BYD', 'CADILLAC', 'CATERHAM', 'CHERY', 'CHEVROLET', 'CITROEN', 'CUPRA', 'DACIA', 'DFSK', 'DFSK MOTOR', 'DFSK SERES SOKON', 'DFSK SERES SOKON FENGON', 'DODGE', 'DONGFENG', 'DS', 'FERRARI', 'FIAT']
...

vehicle_category_type (2 modalités)
['M1', 'N1']

fuel_type (9 modalités)
['diesel', 'diesel/electric', 'e85', 'electric', 'hydrogen', 'lpg', 'ng', 'petrol', 'petrol/electric']

fuel_mode (6 modalités)
['B', 'E', 'F', 'H', 'M', 'P']


### 9.1 Classification nominale / ordinale

L'inspection des modalités montre qu'aucune des variables catégorielles
conservées ne possède un ordre métier naturel exploitable pour la modélisation :

- `manufacturer_make` représente des marques automobiles ;
- `vehicle_category_type` distingue des catégories réglementaires (`M1`, `N1`)
  sans relation d'ordre ;
- `fuel_type` représente différents types de carburant ou d'énergie ;
- `fuel_mode` représente différents modes énergétiques.

Ces quatre variables sont donc traitées comme des variables
**catégorielles nominales**.

Aucune variable catégorielle ordinale n'est identifiée dans le dataset final.

In [11]:
NOMINAL_COLUMNS = [
    "manufacturer_make",
    "vehicle_category_type",
    "fuel_type",
    "fuel_mode",
]

ORDINAL_COLUMNS = []

# Contrôle de cohérence
detected_categorical_columns = set(categorical_columns)

classified_categorical_columns = (
    set(NOMINAL_COLUMNS)
    | set(ORDINAL_COLUMNS)
)

if detected_categorical_columns != classified_categorical_columns:
    raise ValueError(
        "La classification nominale/ordinale "
        "ne correspond pas aux variables catégorielles détectées."
    )

print(f"Variables nominales : {len(NOMINAL_COLUMNS)}")

for column in NOMINAL_COLUMNS:
    print(f"  - {column}")

print(
    f"\nVariables ordinales : "
    f"{len(ORDINAL_COLUMNS)}"
)

Variables nominales : 4
  - manufacturer_make
  - vehicle_category_type
  - fuel_type
  - fuel_mode

Variables ordinales : 0


## 10. Analyse et traitement des valeurs manquantes

### Objectif

Avant de construire les pipelines de preprocessing, cette section analyse les
valeurs manquantes des variables explicatives.

L'objectif est de distinguer :

- les valeurs manquantes probablement accidentelles ou liées à une absence de
  saisie ;
- les valeurs manquantes pouvant avoir une signification métier ;
- les variables pour lesquelles une imputation simple est adaptée ;
- les variables pour lesquelles un indicateur binaire explicite peut être
  pertinent.

Les décisions de traitement sont prises à partir de `X_train` uniquement afin
de ne pas utiliser d'information provenant du jeu de test.

Aucune imputation n'est réalisée dans cette première étape.

### 10.1 Diagnostic quantitatif des valeurs manquantes

Cette première étape mesure la présence de valeurs manquantes dans les
variables explicatives de `X_train`.

Elle permet :

- d'identifier les variables concernées ;
- de mesurer le nombre et le pourcentage de valeurs manquantes ;
- de repérer les variables nécessitant une analyse spécifique avant
  imputation.

Ce diagnostic est uniquement quantitatif.

Il ne permet pas, à lui seul, de déterminer si une valeur est manquante pour
une raison métier ou à cause d'une absence de saisie.

Pour les variables présentant une proportion significative de valeurs
manquantes, cette distinction sera étudiée dans l'étape suivante par
croisement avec les caractéristiques métier du véhicule.

Aucune imputation ni modification de `X_train` n'est réalisée à cette étape.

In [12]:
missing_diagnostic = []

for column in X_train.columns:
    missing_count = X_train[column].isna().sum()
    missing_pct = (
        missing_count
        / len(X_train)
        * 100
    )

    row = {
        "variable": column,
        "dtype": str(X_train[column].dtype),
        "missing_count": missing_count,
        "missing_pct": round(missing_pct, 2),
        "n_unique": X_train[column].nunique(
            dropna=True
        ),
    }

    if pd.api.types.is_numeric_dtype(
        X_train[column]
    ):
        row.update(
            {
                "min": X_train[column].min(),
                "median": X_train[column].median(),
                "mean": X_train[column].mean(),
                "max": X_train[column].max(),
            }
        )

    missing_diagnostic.append(row)


missing_diagnostic_df = (
    pd.DataFrame(missing_diagnostic)
    .sort_values(
        by="missing_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(missing_diagnostic_df)

,variable,dtype,missing_count,missing_pct,n_unique,min,median,mean,max
0,electric_range_km,float64,63215,79.02,532,11.0,3.940000e+02,325.342687,796.00
1,electric_energy_consumption_wh_km,float64,63196,79.00,228,44.0,1.670000e+02,174.957570,594.00
2,co2_reduction_wltp_g_km,float64,35042,43.80,189,0.5,1.700000e+00,1.499348,2.78
3,fuel_consumption,float64,11677,14.60,146,0.3,5.500000e+00,5.498207,17.30
4,engine_capacity_cm3,float64,11355,14.19,104,875.0,1.498000e+03,1576.029922,6749.00
5,wltp_test_mass_kg,float64,225,0.28,1937,733.0,1.609000e+03,1686.848549,3093.00
6,fuel_mode,object,0,0.00,6,NaN,NaN,NaN,NaN
7,fuel_type,object,0,0.00,9,NaN,NaN,NaN,NaN
8,mass_running_order_kg,float64,0,0.00,1339,668.0,1.502000e+03,1566.452712,3085.00
9,vehicle_category_type,object,0,0.00,2,NaN,NaN,NaN,NaN


### 10.2 Analyse métier des valeurs manquantes

Le diagnostic quantitatif précédent permet d'identifier les variables
présentant des valeurs manquantes, mais il ne permet pas d'en déterminer
la cause.

Pour certaines caractéristiques techniques, une valeur manquante peut être
liée à la nature même du véhicule plutôt qu'à une erreur ou à une absence
de saisie.

Cette étape étudie donc les variables numériques contenant des valeurs
manquantes en croisant leur absence avec `fuel_type`.

L'objectif est notamment de vérifier si les valeurs manquantes sont
concentrées sur certains types de motorisation.

Cette analyse permettra ensuite de distinguer, lorsque les données le
permettent :

- les valeurs manquantes ayant vraisemblablement une origine structurelle
  ou métier ;
- les valeurs manquantes ne présentant pas de relation métier évidente
  avec le type de motorisation.

Aucune imputation et aucune modification de `X_train` ne sont réalisées
à cette étape.

In [13]:
# Identification des variables numériques contenant des valeurs manquantes
numeric_columns_with_missing = [
    column
    for column in numeric_columns
    if X_train[column].isna().any()
]

print(
    f"Variables numériques avec valeurs manquantes : "
    f"{len(numeric_columns_with_missing)}"
)

for column in numeric_columns_with_missing:
    print(f"  - {column}")


# Analyse du taux de valeurs manquantes selon le type de motorisation
for column in numeric_columns_with_missing:

    analysis = (
        X_train
        .assign(is_missing=X_train[column].isna())
        .groupby(
            "fuel_type",
            dropna=False,
        )
        .agg(
            observations=("is_missing", "size"),
            missing_count=("is_missing", "sum"),
            missing_pct=("is_missing", "mean"),
        )
        .reset_index()
    )

    analysis["missing_pct"] = (
        analysis["missing_pct"] * 100
    ).round(2)

    print("\n" + "=" * 100)
    print(f"Variable analysée : {column}")
    print("=" * 100)

    display(analysis)

Variables numériques avec valeurs manquantes : 7
  - wltp_test_mass_kg
  - engine_capacity_cm3
  - engine_power_kw
  - electric_energy_consumption_wh_km
  - co2_reduction_wltp_g_km
  - fuel_consumption
  - electric_range_km

Variable analysée : wltp_test_mass_kg


,fuel_type,observations,missing_count,missing_pct
0,diesel,13284,26,0.20
1,diesel/electric,378,1,0.26
2,e85,445,0,0.00
3,electric,11345,102,0.90
4,hydrogen,10,0,0.00
5,lpg,1235,1,0.08
6,ng,8,0,0.00
7,petrol,48081,80,0.17
8,petrol/electric,5214,15,0.29



Variable analysée : engine_capacity_cm3


,fuel_type,observations,missing_count,missing_pct
0,diesel,13284,0,0.0
1,diesel/electric,378,0,0.0
2,e85,445,0,0.0
3,electric,11345,11345,100.0
4,hydrogen,10,10,100.0
5,lpg,1235,0,0.0
6,ng,8,0,0.0
7,petrol,48081,0,0.0
8,petrol/electric,5214,0,0.0



Variable analysée : engine_power_kw


...

,fuel_type,observations,missing_count,missing_pct
0,diesel,13284,4537,34.15
1,diesel/electric,378,378,100.00
2,e85,445,139,31.24
3,electric,11345,11345,100.00
4,hydrogen,10,10,100.00
5,lpg,1235,0,0.00
6,ng,8,6,75.00
7,petrol,48081,13413,27.90
8,petrol/electric,5214,5214,100.00



Variable analysée : fuel_consumption


,fuel_type,observations,missing_count,missing_pct
0,diesel,13284,61,0.46
1,diesel/electric,378,1,0.26
2,e85,445,0,0.00
3,electric,11345,11345,100.00
4,hydrogen,10,10,100.00
5,lpg,1235,3,0.24
6,ng,8,8,100.00
7,petrol,48081,219,0.46
8,petrol/electric,5214,30,0.58



Variable analysée : electric_range_km


,fuel_type,observations,missing_count,missing_pct
0,diesel,13284,13284,100.00
1,diesel/electric,378,0,0.00
2,e85,445,445,100.00
3,electric,11345,129,1.14
4,hydrogen,10,10,100.00
5,lpg,1235,1235,100.00
6,ng,8,8,100.00
7,petrol,48081,48081,100.00
8,petrol/electric,5214,23,0.44


### 10.3 Interprétation métier des valeurs manquantes

Cette section génère automatiquement l'interprétation des résultats obtenus
précédemment afin que les valeurs affichées restent cohérentes avec les données
effectivement chargées dans le notebook.

In [14]:
from IPython.display import Markdown, display


# ---------------------------------------------------------------------
# Fonctions utilitaires
# ---------------------------------------------------------------------

def get_missing_pct(column, fuel_type):
    """
    Calcule le pourcentage de valeurs manquantes d'une variable
    pour un type de motorisation donné dans X_train.
    """

    mask = X_train["fuel_type"].eq(fuel_type)

    if mask.sum() == 0:
        return None

    return (
        X_train.loc[mask, column]
        .isna()
        .mean()
        * 100
    )


def pct(column, fuel_type):
    """
    Retourne le pourcentage de valeurs manquantes formaté
    pour l'affichage Markdown.
    """

    value = get_missing_pct(
        column,
        fuel_type,
    )

    if value is None:
        return "non observé"

    return f"{value:.2f} %"


# ---------------------------------------------------------------------
# Génération dynamique de l'interprétation
# ---------------------------------------------------------------------

interpretation = f"""
#### `wltp_test_mass_kg` — Masse du véhicule lors du test WLTP

Le taux de valeurs manquantes reste très faible pour les principales
motorisations :

- diesel : **{pct("wltp_test_mass_kg", "diesel")}** ;
- diesel/électrique : **{pct("wltp_test_mass_kg", "diesel/electric")}** ;
- électrique : **{pct("wltp_test_mass_kg", "electric")}** ;
- essence : **{pct("wltp_test_mass_kg", "petrol")}** ;
- essence/électrique : **{pct("wltp_test_mass_kg", "petrol/electric")}**.

**Interprétation :** l'absence de la masse WLTP reste marginale et aucune
signification métier structurelle évidente n'est mise en évidence par le
croisement avec le type de motorisation.


#### `engine_capacity_cm3` — Cylindrée du moteur thermique

Le comportement de cette variable est particulièrement net :

- véhicules électriques :
  **{pct("engine_capacity_cm3", "electric")}** de valeurs manquantes ;
- véhicules à hydrogène :
  **{pct("engine_capacity_cm3", "hydrogen")}** de valeurs manquantes ;
- diesel : **{pct("engine_capacity_cm3", "diesel")}** ;
- essence : **{pct("engine_capacity_cm3", "petrol")}** ;
- essence/électrique :
  **{pct("engine_capacity_cm3", "petrol/electric")}**.

**Interprétation :** lorsque les valeurs manquantes sont concentrées sur les
véhicules électriques et à hydrogène, l'absence de cylindrée présente une
forte composante structurelle liée au type de motorisation.

Une imputation globale par la médiane serait alors inadaptée, car elle
attribuerait artificiellement une cylindrée de moteur thermique à des
véhicules pour lesquels cette caractéristique n'est pas renseignée.


#### `engine_power_kw` — Puissance moteur

Taux de valeurs manquantes observé pour les principales motorisations :

- diesel : **{pct("engine_power_kw", "diesel")}** ;
- électrique : **{pct("engine_power_kw", "electric")}** ;
- essence : **{pct("engine_power_kw", "petrol")}** ;
- essence/électrique :
  **{pct("engine_power_kw", "petrol/electric")}**.

**Interprétation :** lorsque ces taux restent extrêmement faibles, les valeurs
manquantes sont marginales et aucune structure métier particulière n'est mise
en évidence par cette analyse.


#### `electric_energy_consumption_wh_km` — Consommation d'énergie électrique

Taux de valeurs manquantes :

- diesel :
  **{pct("electric_energy_consumption_wh_km", "diesel")}** ;
- E85 :
  **{pct("electric_energy_consumption_wh_km", "e85")}** ;
- électrique :
  **{pct("electric_energy_consumption_wh_km", "electric")}** ;
- hydrogène :
  **{pct("electric_energy_consumption_wh_km", "hydrogen")}** ;
- GPL :
  **{pct("electric_energy_consumption_wh_km", "lpg")}** ;
- gaz naturel :
  **{pct("electric_energy_consumption_wh_km", "ng")}** ;
- essence :
  **{pct("electric_energy_consumption_wh_km", "petrol")}** ;
- diesel/électrique :
  **{pct("electric_energy_consumption_wh_km", "diesel/electric")}** ;
- essence/électrique :
  **{pct("electric_energy_consumption_wh_km", "petrol/electric")}**.

**Interprétation :** lorsque l'absence de cette information se concentre sur
les motorisations non électriques tandis que la variable est majoritairement
renseignée pour les véhicules électriques ou hybrides, les valeurs manquantes
présentent une forte composante structurelle.

Une imputation globale par la médiane serait alors peu adaptée.


#### `co2_reduction_wltp_g_km` — Réduction des émissions de CO₂ WLTP

Taux de valeurs manquantes :

- diesel :
  **{pct("co2_reduction_wltp_g_km", "diesel")}** ;
- diesel/électrique :
  **{pct("co2_reduction_wltp_g_km", "diesel/electric")}** ;
- E85 :
  **{pct("co2_reduction_wltp_g_km", "e85")}** ;
- électrique :
  **{pct("co2_reduction_wltp_g_km", "electric")}** ;
- hydrogène :
  **{pct("co2_reduction_wltp_g_km", "hydrogen")}** ;
- GPL :
  **{pct("co2_reduction_wltp_g_km", "lpg")}** ;
- gaz naturel :
  **{pct("co2_reduction_wltp_g_km", "ng")}** ;
- essence :
  **{pct("co2_reduction_wltp_g_km", "petrol")}** ;
- essence/électrique :
  **{pct("co2_reduction_wltp_g_km", "petrol/electric")}**.

**Interprétation :** la présence ou l'absence de cette variable ne doit pas
être interprétée directement comme le niveau global de réduction des émissions
de CO₂ du véhicule.

Les résultats montrent notamment que cette information peut être absente pour
l'ensemble des observations de certaines motorisations dans l'échantillon
étudié.

Cette absence systématique constitue bien une structure dans les données.
Cependant, une valeur manquante ne signifie ni que la réduction de CO₂ est
égale à zéro, ni qu'elle est nécessairement élevée.

Pour d'autres motorisations, la coexistence de valeurs présentes et manquantes
montre également que le seul `fuel_type` ne suffit pas à déterminer une valeur
de remplacement pertinente.

**Conclusion :** `co2_reduction_wltp_g_km` doit faire l'objet d'un traitement
spécifique. Une imputation globale par la médiane ne sera pas appliquée
automatiquement à cette variable.


#### `fuel_consumption` — Consommation de carburant

Taux de valeurs manquantes :

- diesel :
  **{pct("fuel_consumption", "diesel")}** ;
- diesel/électrique :
  **{pct("fuel_consumption", "diesel/electric")}** ;
- E85 :
  **{pct("fuel_consumption", "e85")}** ;
- électrique :
  **{pct("fuel_consumption", "electric")}** ;
- hydrogène :
  **{pct("fuel_consumption", "hydrogen")}** ;
- GPL :
  **{pct("fuel_consumption", "lpg")}** ;
- gaz naturel :
  **{pct("fuel_consumption", "ng")}** ;
- essence :
  **{pct("fuel_consumption", "petrol")}** ;
- essence/électrique :
  **{pct("fuel_consumption", "petrol/electric")}**.

**Interprétation :** lorsque les valeurs manquantes sont principalement
concentrées sur les motorisations pour lesquelles la consommation de carburant
n'est pas applicable ou n'est pas renseignée de la même manière, une forte
composante structurelle est mise en évidence.

Les faibles taux résiduels éventuellement observés sur les motorisations
utilisant effectivement du carburant doivent cependant être distingués de
cette absence structurelle.


#### `electric_range_km` — Autonomie électrique

Taux de valeurs manquantes :

- diesel :
  **{pct("electric_range_km", "diesel")}** ;
- diesel/électrique :
  **{pct("electric_range_km", "diesel/electric")}** ;
- E85 :
  **{pct("electric_range_km", "e85")}** ;
- électrique :
  **{pct("electric_range_km", "electric")}** ;
- hydrogène :
  **{pct("electric_range_km", "hydrogen")}** ;
- GPL :
  **{pct("electric_range_km", "lpg")}** ;
- gaz naturel :
  **{pct("electric_range_km", "ng")}** ;
- essence :
  **{pct("electric_range_km", "petrol")}** ;
- essence/électrique :
  **{pct("electric_range_km", "petrol/electric")}**.

**Interprétation :** lorsque l'autonomie électrique est absente pour les
motorisations non électriques mais majoritairement renseignée pour les
véhicules électriques et hybrides, les valeurs manquantes présentent une
forte composante structurelle.

Les éventuelles valeurs manquantes résiduelles observées chez les véhicules
électriques ou hybrides doivent néanmoins être considérées séparément.


### Synthèse

L'analyse permet de distinguer trois situations :

1. **Valeurs manquantes présentant une forte composante structurelle**
   - `engine_capacity_cm3` ;
   - `electric_energy_consumption_wh_km` ;
   - `fuel_consumption` ;
   - `electric_range_km`.

2. **Valeurs manquantes rares sans structure métier évidente mise en évidence**
   - `wltp_test_mass_kg` ;
   - `engine_power_kw`.

3. **Valeurs manquantes présentant une structure plus complexe**
   - `co2_reduction_wltp_g_km`.

Ces résultats montrent qu'une stratégie d'imputation numérique unique,
appliquée indistinctement à toutes les variables, serait insuffisante.

La prochaine étape consistera à définir, variable par variable, une stratégie
de traitement cohérente avec la nature des valeurs manquantes observées avant
la construction du pipeline de preprocessing.
"""


# ---------------------------------------------------------------------
# Affichage du Markdown généré
# ---------------------------------------------------------------------

display(
    Markdown(interpretation)
)


#### `wltp_test_mass_kg` — Masse du véhicule lors du test WLTP

Le taux de valeurs manquantes reste très faible pour les principales
motorisations :

- diesel : **0.20 %** ;
- diesel/électrique : **0.26 %** ;
- électrique : **0.90 %** ;
- essence : **0.17 %** ;
- essence/électrique : **0.29 %**.

**Interprétation :** l'absence de la masse WLTP reste marginale et aucune
signification métier structurelle évidente n'est mise en évidence par le
croisement avec le type de motorisation.


#### `engine_capacity_cm3` — Cylindrée du moteur thermique

Le comportement de cette variable est particulièrement net :

- véhicules électriques :
  **100.00 %** de valeurs manquantes ;
- véhicules à hydrogène :
  **100.00 %** de valeurs manquantes ;
- diesel : **0.00 %** ;
- essence : **0.00 %** ;
- essence/électrique :
  **0.00 %**.

**Interprétation :** lorsque les valeurs manquantes sont concentrées sur les
véhicules électriques et à hydrogène, l'absence de cylindrée présente une
forte composante structurelle liée au type de motorisation.

Une imputation globale par la médiane serait alors inadaptée, car elle
attribuerait artificiellement une cylindrée de moteur thermique à des
véhicules pour lesquels cette caractéristique n'est pas renseignée.


#### `engine_power_kw` — Puissance moteur

Taux de valeurs manquantes observé pour les principales motorisations :

- diesel : **0.00 %** ;
- électrique : **0.00 %** ;
- essence : **0.00 %** ;
- essence/électrique :
  **0.00 %**.

**Interprétation :** lorsque ces taux restent extrêmement faibles, les valeurs
manquantes sont marginales et aucune structure métier particulière n'est mise
en évidence par cette analyse.


#### `electric_energy_consumption_wh_km` — Consommation d'énergie électrique

Taux de valeurs manquantes :

- diesel :
  **100.00 %** ;
- E85 :
  **100.00 %** ;
- électrique :
  **0.96 %** ;
- hydrogène :
  **100.00 %** ;
- GPL :
  **100.00 %** ;
- gaz naturel :
  **100.00 %** ;
- essence :
  **100.00 %** ;
- diesel/électrique :
  **0.26 %** ;
- essence/électrique :
  **0.44 %**.

**Interprétation :** lorsque l'absence de cette information se concentre sur
les motorisations non électriques tandis que la variable est majoritairement
renseignée pour les véhicules électriques ou hybrides, les valeurs manquantes
présentent une forte composante structurelle.

Une imputation globale par la médiane serait alors peu adaptée.


#### `co2_reduction_wltp_g_km` — Réduction des émissions de CO₂ WLTP

Taux de valeurs manquantes :

- diesel :
  **34.15 %** ;
- diesel/électrique :
  **100.00 %** ;
- E85 :
  **31.24 %** ;
- électrique :
  **100.00 %** ;
- hydrogène :
  **100.00 %** ;
- GPL :
  **0.00 %** ;
- gaz naturel :
  **75.00 %** ;
- essence :
  **27.90 %** ;
- essence/électrique :
  **100.00 %**.

**Interprétation :** la présence ou l'absence de cette variable ne doit pas
être interprétée directement comme le niveau global de réduction des émissions
de CO₂ du véhicule.

Les résultats montrent notamment que cette information peut être absente pour
l'ensemble des observations de certaines motorisations dans l'échantillon
étudié.

Cette absence systématique constitue bien une structure dans les données.
Cependant, une valeur manquante ne signifie ni que la réduction de CO₂ est
égale à zéro, ni qu'elle est nécessairement élevée.

Pour d'autres motorisations, la coexistence de valeurs présentes et manquantes
montre également que le seul `fuel_type` ne suffit pas à déterminer une valeur
de remplacement pertinente.

**Conclusion :** `co2_reduction_wltp_g_km` doit faire l'objet d'un traitement
spécifique. Une imputation globale par la médiane ne sera pas appliquée
automatiquement à cette variable.


#### `fuel_consumption` — Consommation de carburant

Taux de valeurs manquantes :

- diesel :
  **0.46 %** ;
- diesel/électrique :
  **0.26 %** ;
- E85 :
  **0.00 %** ;
- électrique :
  **100.00 %** ;
- hydrogène :
  **100.00 %** ;
- GPL :
  **0.24 %** ;
- gaz naturel :
  **100.00 %** ;
- essence :
  **0.46 %** ;
- essence/électrique :
  **0.58 %**.

**Interprétation :** lorsque les valeurs manquantes sont principalement
concentrées sur les motorisations pour lesquelles la consommation de carburant
n'est pas applicable ou n'est pas renseignée de la même manière, une forte
composante structurelle est mise en évidence.

Les faibles taux résiduels éventuellement observés sur les motorisations
utilisant effectivement du carburant doivent cependant être distingués de
cette absence structurelle.


#### `electric_range_km` — Autonomie électrique

Taux de valeurs manquantes :

- diesel :
  **100.00 %** ;
- diesel/électrique :
  **0.00 %** ;
- E85 :
  **100.00 %** ;
- électrique :
  **1.14 %** ;
- hydrogène :
  **100.00 %** ;
- GPL :
  **100.00 %** ;
- gaz naturel :
  **100.00 %** ;
- essence :
  **100.00 %** ;
- essence/électrique :
  **0.44 %**.

**Interprétation :** lorsque l'autonomie électrique est absente pour les
motorisations non électriques mais majoritairement renseignée pour les
véhicules électriques et hybrides, les valeurs manquantes présentent une
forte composante structurelle.

Les éventuelles valeurs manquantes résiduelles observées chez les véhicules
électriques ou hybrides doivent néanmoins être considérées séparément.


### Synthèse

L'analyse permet de distinguer trois situations :

1. **Valeurs manquantes présentant une forte composante structurelle**
   - `engine_capacity_cm3` ;
   - `electric_energy_consumption_wh_km` ;
   - `fuel_consumption` ;
   - `electric_range_km`.

2. **Valeurs manquantes rares sans structure métier évidente mise en évidence**
   - `wltp_test_mass_kg` ;
   - `engine_power_kw`.

3. **Valeurs manquantes présentant une structure plus complexe**
   - `co2_reduction_wltp_g_km`.

Ces résultats montrent qu'une stratégie d'imputation numérique unique,
appliquée indistinctement à toutes les variables, serait insuffisante.

La prochaine étape consistera à définir, variable par variable, une stratégie
de traitement cohérente avec la nature des valeurs manquantes observées avant
la construction du pipeline de preprocessing.


### 10.4 Définition des stratégies de traitement des valeurs manquantes

Les analyses précédentes montrent que toutes les valeurs manquantes ne
possèdent pas la même signification.

Une imputation uniforme de toutes les variables numériques par leur médiane
n'est donc pas retenue.

Pour certaines variables techniques, deux types de valeurs manquantes doivent
être distingués :

- **NaN structurel** : la caractéristique n'est pas applicable au véhicule
  considéré. Dans ce cas, l'absence de valeur possède une signification métier.
- **NaN résiduel** : la caractéristique est applicable au véhicule, mais sa
  valeur reste manquante. Il s'agit alors d'une donnée absente qui nécessite
  une imputation.

Lorsqu'un NaN est structurel et que la valeur `0` possède une interprétation
métier cohérente, le NaN peut être remplacé par `0`.

Lorsqu'un NaN est résiduel, une imputation statistique est réalisée. Lorsque
cela est pertinent, un indicateur binaire `has_...` permet également de
conserver l'information selon laquelle la valeur originale était présente
ou absente.

Les statistiques nécessaires aux imputations seront apprises exclusivement
sur `X_train`.


#### 1. `engine_capacity_cm3` — Cylindrée du moteur thermique

La cylindrée correspond à la capacité du moteur thermique.

L'analyse précédente montre que les valeurs manquantes sont concentrées sur
les véhicules électriques et à hydrogène, tandis qu'aucune valeur manquante
n'est observée pour les autres motorisations dans l'échantillon étudié.

Pour les véhicules électriques et à hydrogène, l'absence de cylindrée est
considérée comme **structurelle**.

**Décision :**

- pour une motorisation pour laquelle la cylindrée thermique n'est pas
  applicable, remplacer le NaN par `0` ;
- si de futurs jeux de données présentent un NaN pour une motorisation
  thermique, celui-ci sera considéré comme **résiduel** et fera l'objet
  d'une imputation statistique ;
- aucun indicateur binaire supplémentaire n'est créé à ce stade, car
  `fuel_type` permet déjà d'identifier la nature de la motorisation.

La valeur `0` représente ici l'absence de cylindrée thermique et non une
cylindrée inconnue.


#### 2. `electric_energy_consumption_wh_km` — Consommation d'énergie électrique

Cette variable mesure la consommation d'énergie électrique du véhicule.

L'analyse montre que son absence est systématique pour les motorisations
non électriques, tandis que la variable est presque toujours renseignée
pour les véhicules électriques ou hybrides.

Deux situations doivent donc être distinguées :

- pour une motorisation non électrique, le NaN est **structurel** ;
- pour un véhicule électrique ou hybride, le NaN est **résiduel**.

**Décision :**

- remplacer les NaN structurels par `0` ;
- imputer statistiquement les NaN résiduels des véhicules électriques
  ou hybrides ;
- créer un indicateur binaire `has_electric_energy_consumption` permettant,
  pour les véhicules auxquels cette caractéristique est applicable, de
  conserver l'information selon laquelle la valeur originale était présente
  ou manquante.

La valeur `0` représente l'absence de consommation électrique applicable
à la motorisation.

Elle ne doit pas être utilisée pour représenter une consommation électrique
inconnue d'un véhicule électrique ou hybride.


#### 3. `electric_range_km` — Autonomie électrique

Cette variable représente l'autonomie électrique du véhicule.

Une motorisation ne disposant pas d'une capacité de déplacement électrique
n'a pas d'autonomie électrique applicable.

L'analyse confirme cette structure : les motorisations non électriques
présentent une absence systématique de cette information, alors que la
variable est presque toujours renseignée pour les véhicules électriques
et hybrides.

Deux situations sont donc distinguées :

- pour une motorisation non électrique, le NaN est **structurel** ;
- pour un véhicule électrique ou hybride, le NaN est **résiduel**.

**Décision :**

- remplacer les NaN structurels par `0` ;
- imputer statistiquement les NaN résiduels des véhicules électriques
  ou hybrides ;
- créer un indicateur binaire `has_electric_range` afin de conserver
  l'information selon laquelle l'autonomie électrique originale était
  présente ou manquante lorsque cette caractéristique est applicable.

Il ne serait pas cohérent d'attribuer à un véhicule purement thermique
l'autonomie électrique médiane d'un véhicule électrique.

Inversement, remplacer par `0` une autonomie manquante d'un véhicule
électrique reviendrait à lui attribuer artificiellement une autonomie
électrique de 0 km. Ces NaN résiduels doivent donc être imputés.


#### 4. `fuel_consumption` — Consommation de carburant

Cette variable présente également deux situations différentes.

Pour certaines motorisations, l'absence de consommation de carburant est
liée à la nature du véhicule. Pour d'autres motorisations utilisant
effectivement du carburant, de faibles proportions de valeurs manquantes
subsistent.

Deux types de NaN sont donc distingués :

- **NaN structurel** lorsque la consommation de carburant n'est pas
  applicable à la motorisation ;
- **NaN résiduel** lorsque le véhicule utilise un carburant mais que sa
  consommation n'est pas renseignée.

**Décision :**

- remplacer les NaN structurels par `0` ;
- imputer statistiquement les NaN résiduels uniquement parmi les
  motorisations pour lesquelles la consommation de carburant est applicable ;
- créer un indicateur binaire `has_fuel_consumption` permettant de conserver
  l'information selon laquelle la consommation originale était présente ou
  manquante lorsque cette caractéristique est applicable.

Cette distinction évite d'attribuer artificiellement une consommation de
carburant à un véhicule pour lequel cette caractéristique n'est pas
applicable.


#### 5. `wltp_test_mass_kg` — Masse du véhicule lors du test WLTP

Les valeurs manquantes sont rares et aucune justification métier structurelle
n'a été mise en évidence par l'analyse précédente.

La masse WLTP est une caractéristique applicable au véhicule indépendamment
du type de motorisation.

Les valeurs manquantes sont donc considérées comme **résiduelles**.

**Décision :**

- appliquer une imputation statistique par la médiane calculée exclusivement
  à partir de `X_train` ;
- ne pas créer d'indicateur binaire spécifique compte tenu du faible taux
  de valeurs manquantes et de l'absence de structure métier mise en évidence.

La médiane est retenue afin de limiter la sensibilité de l'imputation aux
valeurs extrêmes.


#### 6. `engine_power_kw` — Puissance moteur

Les valeurs manquantes observées sont extrêmement rares et aucune structure
métier particulière n'a été identifiée.

La puissance moteur reste une caractéristique applicable aux véhicules
concernés.

Les valeurs manquantes sont donc considérées comme **résiduelles**.

**Décision :**

- appliquer une imputation par la médiane calculée exclusivement à partir
  de `X_train` ;
- ne pas créer d'indicateur binaire spécifique compte tenu du caractère
  exceptionnel des valeurs manquantes.


#### 7. `co2_reduction_wltp_g_km` — Réduction des émissions de CO₂ WLTP

Cette variable présente une proportion importante de valeurs manquantes et
une structure particulière.

Une imputation globale par la moyenne ou la médiane n'est pas retenue, car
elle attribuerait artificiellement une valeur de réduction de CO₂ aux
observations pour lesquelles aucune réduction n'est renseignée dans cette
variable.

L'absence de valeur constitue elle-même une information potentiellement
utile au modèle.

**Décision :**

- créer une variable binaire `has_co2_reduction_wltp` :
  - `1` lorsque la valeur originale de `co2_reduction_wltp_g_km`
    est renseignée ;
  - `0` lorsque la valeur originale est manquante ;
- remplacer ensuite les valeurs manquantes de
  `co2_reduction_wltp_g_km` par `0`.

La valeur `0` utilisée pour l'imputation ne signifie pas nécessairement
que le véhicule ne bénéficie physiquement d'aucune réduction de ses
émissions de CO₂.

Elle représente l'absence de réduction renseignée dans cette variable.

L'indicateur `has_co2_reduction_wltp` permet de distinguer une valeur
réellement renseignée d'une valeur créée lors du traitement des données.


### Synthèse des décisions

| Variable | Nature des NaN | Traitement retenu | Indicateur |
|---|---|---|---|
| `engine_capacity_cm3` | Structurelle si cylindrée non applicable ; éventuelle absence résiduelle sinon | Structurel → `0` ; résiduel → imputation statistique | Aucun à ce stade |
| `electric_energy_consumption_wh_km` | Structurelle pour les motorisations non électriques ; résiduelle pour les électriques/hybrides | Structurel → `0` ; résiduel → imputation statistique | `has_electric_energy_consumption` |
| `electric_range_km` | Structurelle pour les motorisations non électriques ; résiduelle pour les électriques/hybrides | Structurel → `0` ; résiduel → imputation statistique | `has_electric_range` |
| `fuel_consumption` | Structurelle lorsque non applicable ; résiduelle lorsqu'elle est applicable | Structurel → `0` ; résiduel → imputation statistique | `has_fuel_consumption` |
| `wltp_test_mass_kg` | Rare et résiduelle | Médiane apprise sur `X_train` | Aucun |
| `engine_power_kw` | Exceptionnelle et résiduelle | Médiane apprise sur `X_train` | Aucun |
| `co2_reduction_wltp_g_km` | Absence informative / structure particulière | NaN → `0` | `has_co2_reduction_wltp` |


### Principe général retenu

La stratégie de preprocessing distingue désormais trois situations.

**1. NaN structurel**

La caractéristique n'est pas applicable au véhicule.

Lorsque la valeur `0` possède une interprétation métier cohérente, elle est
utilisée pour représenter cette non-applicabilité.

**2. NaN résiduel**

La caractéristique est applicable au véhicule, mais sa valeur n'a pas été
renseignée.

Une imputation statistique est alors réalisée à partir de `X_train`.

Pour les variables où cette absence peut constituer une information utile,
un indicateur `has_...` permet de conserver la connaissance de l'état
original de la donnée.

**3. Absence informative particulière**

Pour `co2_reduction_wltp_g_km`, la valeur manquante est remplacée par `0`
tout en conservant explicitement son absence originale au moyen de
`has_co2_reduction_wltp`.


### Variables binaires supplémentaires retenues

Le preprocessing introduira donc les variables suivantes :

- `has_electric_energy_consumption` ;
- `has_electric_range` ;
- `has_fuel_consumption` ;
- `has_co2_reduction_wltp`.

Ces indicateurs ne sont pas créés systématiquement pour toutes les variables
présentant des valeurs manquantes. Ils sont retenus lorsque la distinction
entre valeur originale présente et valeur imputée apporte une information
potentiellement utile au modèle.

Toutes les statistiques nécessaires aux imputations seront apprises
exclusivement à partir de `X_train`.

Ces paramètres seront ensuite réutilisés sans réapprentissage lors de la
transformation de `X_test`, afin d'éviter toute fuite d'information
(`data leakage`).

### 10.5 Création des indicateurs binaires de présence

Conformément aux décisions prises au point 10.4, un indicateur binaire est
créé pour les variables dont l'absence d'information doit être conservée
comme information supplémentaire pour le modèle.

Pour chaque variable concernée :

- `1` indique que la valeur était présente dans la donnée originale ;
- `0` indique que la valeur était manquante avant traitement.

Les indicateurs sont créés **avant toute imputation**, afin de préserver
l'information originale sur la présence ou l'absence de la donnée.

Les indicateurs retenus sont :

- `has_electric_energy_consumption_wh_km` ;
- `has_electric_range_km` ;
- `has_fuel_consumption` ;
- `has_co2_reduction_wltp_g_km`.

Aucun indicateur supplémentaire n'est créé pour les autres variables,
conformément aux décisions du point 10.4.

In [15]:
# ---------------------------------------------------------------------
# 10.5 - Création des indicateurs binaires de présence
# ---------------------------------------------------------------------

# Copies de travail utilisées pour les traitements à venir.
X_train_processed = X_train.copy()
X_test_processed = X_test.copy()


# Correspondance entre les variables sources et leurs indicateurs.
indicator_columns = {
    "electric_energy_consumption_wh_km":
        "has_electric_energy_consumption_wh_km",

    "electric_range_km":
        "has_electric_range_km",

    "fuel_consumption":
        "has_fuel_consumption",

    "co2_reduction_wltp_g_km":
        "has_co2_reduction_wltp_g_km",
}


# Création des indicateurs AVANT toute imputation.
#
# 1 -> valeur présente dans la donnée originale
# 0 -> valeur manquante dans la donnée originale
for source_column, indicator_column in indicator_columns.items():

    X_train_processed[indicator_column] = (
        X_train_processed[source_column]
        .notna()
        .astype("int8")
    )

    X_test_processed[indicator_column] = (
        X_test_processed[source_column]
        .notna()
        .astype("int8")
    )


# Vérification des indicateurs créés.
for indicator_column in indicator_columns.values():

    train_values = sorted(
        X_train_processed[indicator_column]
        .unique()
        .tolist()
    )

    test_values = sorted(
        X_test_processed[indicator_column]
        .unique()
        .tolist()
    )

    print(
        f"{indicator_column}\n"
        f"  X_train : {train_values}\n"
        f"  X_test  : {test_values}\n"
    )

has_electric_energy_consumption_wh_km
  X_train : [0, 1]
  X_test  : [0, 1]

has_electric_range_km
  X_train : [0, 1]
  X_test  : [0, 1]

has_fuel_consumption
  X_train : [0, 1]
  X_test  : [0, 1]

has_co2_reduction_wltp_g_km
  X_train : [0, 1]
  X_test  : [0, 1]



### 10.6 Traitement conditionnel des NaN structurels et résiduels

Conformément aux décisions prises au point 10.4, les valeurs manquantes
des variables suivantes nécessitent un traitement conditionnel :

- `engine_capacity_cm3` ;
- `electric_energy_consumption_wh_km` ;
- `electric_range_km` ;
- `fuel_consumption`.

Deux situations sont distinguées :

1. **NaN structurel**

   La variable n'est pas applicable au type de motorisation concerné.

   Dans ce cas :

   - la valeur manquante est remplacée par `0` ;
   - aucune médiane n'est utilisée pour cette observation.

2. **NaN résiduel**

   La variable est applicable au véhicule, mais sa valeur est absente.

   Dans ce cas :

   - la médiane est calculée exclusivement à partir des valeurs pertinentes
     et présentes de `X_train` ;
   - cette médiane est utilisée pour imputer les NaN résiduels de `X_train` ;
   - la même médiane apprise sur `X_train` est utilisée pour les NaN
     résiduels de `X_test`.

Les indicateurs binaires créés au point 10.5 restent inchangés et conservent
l'information indiquant si la valeur était initialement présente ou absente.

Cette étape ne traite pas encore :

- `co2_reduction_wltp_g_km` ;
- `wltp_test_mass_kg` ;
- `engine_power_kw`.

Ces variables feront l'objet des étapes suivantes.

In [16]:
# ---------------------------------------------------------------------
# 10.6 - Traitement conditionnel des NaN structurels et résiduels
# ---------------------------------------------------------------------

# Groupes de motorisations nécessaires pour appliquer les décisions
# métier définies au point 10.4.

# Variables électriques applicables aux véhicules électriques
# et hybrides.
electric_fuel_types = {
    "electric",
    "diesel/electric",
    "petrol/electric",
}

# Cylindrée applicable aux véhicules possédant un moteur thermique.
thermal_engine_fuel_types = {
    "diesel",
    "diesel/electric",
    "e85",
    "lpg",
    "ng",
    "petrol",
    "petrol/electric",
}

# Consommation de carburant applicable aux motorisations utilisant
# effectivement un carburant.
fuel_consuming_types = {
    "diesel",
    "diesel/electric",
    "e85",
    "lpg",
    "petrol",
    "petrol/electric",
}


# ---------------------------------------------------------------------
# Fonction de traitement conditionnel
# ---------------------------------------------------------------------

def apply_conditional_imputation(
    train_df,
    test_df,
    column,
    applicable_fuel_types,
):
    """
    Traite les NaN d'une variable selon les décisions du point 10.4.

    NaN structurel :
        variable non applicable à la motorisation -> remplacement par 0.

    NaN résiduel :
        variable applicable mais valeur absente -> remplacement par
        la médiane calculée exclusivement sur X_train.

    La médiane apprise sur X_train est également utilisée pour X_test.
    """

    # La variable est-elle applicable à la motorisation ?
    train_applicable = train_df["fuel_type"].isin(
        applicable_fuel_types
    )

    test_applicable = test_df["fuel_type"].isin(
        applicable_fuel_types
    )


    # -------------------------------------------------------------
    # Identification des NaN AVANT traitement
    # -------------------------------------------------------------

    train_structural_mask = (
        ~train_applicable
        & train_df[column].isna()
    )

    train_residual_mask = (
        train_applicable
        & train_df[column].isna()
    )

    test_structural_mask = (
        ~test_applicable
        & test_df[column].isna()
    )

    test_residual_mask = (
        test_applicable
        & test_df[column].isna()
    )


    # -------------------------------------------------------------
    # Calcul de la médiane exclusivement sur X_train
    #
    # Seules les observations où la variable est applicable
    # et réellement renseignée participent au calcul.
    # -------------------------------------------------------------

    train_reference_values = train_df.loc[
        train_applicable
        & train_df[column].notna(),
        column,
    ]

    if train_reference_values.empty:
        raise ValueError(
            f"Aucune valeur pertinente disponible dans X_train "
            f"pour calculer la médiane de '{column}'."
        )

    median_train = train_reference_values.median()


    # -------------------------------------------------------------
    # Traitement des NaN structurels
    # -------------------------------------------------------------

    train_df.loc[
        train_structural_mask,
        column,
    ] = 0.0

    test_df.loc[
        test_structural_mask,
        column,
    ] = 0.0


    # -------------------------------------------------------------
    # Traitement des NaN résiduels
    # -------------------------------------------------------------

    train_df.loc[
        train_residual_mask,
        column,
    ] = median_train

    test_df.loc[
        test_residual_mask,
        column,
    ] = median_train


    # -------------------------------------------------------------
    # Résultat du traitement
    # -------------------------------------------------------------

    return {
        "variable": column,

        "median_train": median_train,

        "train_structural_nan": int(
            train_structural_mask.sum()
        ),

        "train_residual_nan": int(
            train_residual_mask.sum()
        ),

        "test_structural_nan": int(
            test_structural_mask.sum()
        ),

        "test_residual_nan": int(
            test_residual_mask.sum()
        ),
    }


# ---------------------------------------------------------------------
# Application aux quatre variables concernées
# ---------------------------------------------------------------------

conditional_imputation_report = []


conditional_imputation_report.append(
    apply_conditional_imputation(
        train_df=X_train_processed,
        test_df=X_test_processed,
        column="engine_capacity_cm3",
        applicable_fuel_types=thermal_engine_fuel_types,
    )
)


conditional_imputation_report.append(
    apply_conditional_imputation(
        train_df=X_train_processed,
        test_df=X_test_processed,
        column="electric_energy_consumption_wh_km",
        applicable_fuel_types=electric_fuel_types,
    )
)


conditional_imputation_report.append(
    apply_conditional_imputation(
        train_df=X_train_processed,
        test_df=X_test_processed,
        column="electric_range_km",
        applicable_fuel_types=electric_fuel_types,
    )
)


conditional_imputation_report.append(
    apply_conditional_imputation(
        train_df=X_train_processed,
        test_df=X_test_processed,
        column="fuel_consumption",
        applicable_fuel_types=fuel_consuming_types,
    )
)


# ---------------------------------------------------------------------
# Tableau de contrôle du traitement 10.6
# ---------------------------------------------------------------------

conditional_imputation_report_df = pd.DataFrame(
    conditional_imputation_report
)

display(conditional_imputation_report_df)

,variable,median_train,train_structural_nan,train_residual_nan,test_structural_nan,test_residual_nan
0,engine_capacity_cm3,1498.0,11355,0,2840,0
1,electric_energy_consumption_wh_km,167.0,63063,133,15733,33
2,electric_range_km,394.0,63063,152,15733,39
3,fuel_consumption,5.5,11363,314,2842,90


### 10.7 Traitement de `co2_reduction_wltp_g_km`

Conformément à la décision prise au point 10.4, la variable
`co2_reduction_wltp_g_km` fait l'objet d'un traitement spécifique.

Pour cette variable, l'absence de valeur est considérée comme une information
particulière qui ne doit pas être traitée par une imputation statistique.

La stratégie retenue est donc :

- les valeurs présentes sont conservées telles quelles ;
- les valeurs `NaN` sont remplacées par `0` ;
- aucune moyenne ou médiane n'est calculée ;
- l'information indiquant si la valeur était initialement présente ou absente
  est déjà conservée dans `has_co2_reduction_wltp_g_km`, créé au point 10.5.

Ainsi, la valeur numérique traitée et l'information sur sa présence initiale
restent disponibles séparément pour le futur modèle.

In [17]:
# ---------------------------------------------------------------------
# 10.7 - Traitement spécifique de co2_reduction_wltp_g_km
# ---------------------------------------------------------------------

column = "co2_reduction_wltp_g_km"


# ---------------------------------------------------------------------
# 1. Identification des NaN AVANT traitement
# ---------------------------------------------------------------------

train_missing_mask = (
    X_train_processed[column]
    .isna()
)

test_missing_mask = (
    X_test_processed[column]
    .isna()
)

train_missing_count = int(
    train_missing_mask.sum()
)

test_missing_count = int(
    test_missing_mask.sum()
)


# ---------------------------------------------------------------------
# 2. Application de la décision du point 10.4
#
# NaN -> 0
#
# L'indicateur has_co2_reduction_wltp_g_km a déjà été créé
# au point 10.5 avant toute imputation.
# ---------------------------------------------------------------------

X_train_processed.loc[
    train_missing_mask,
    column,
] = 0.0

X_test_processed.loc[
    test_missing_mask,
    column,
] = 0.0


# ---------------------------------------------------------------------
# 3. Vérification après traitement
# ---------------------------------------------------------------------

train_remaining_nan = int(
    X_train_processed[column]
    .isna()
    .sum()
)

test_remaining_nan = int(
    X_test_processed[column]
    .isna()
    .sum()
)


# ---------------------------------------------------------------------
# 4. Rapport du traitement effectué
# ---------------------------------------------------------------------

co2_reduction_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train",
            "nan_avant_traitement": train_missing_count,
            "traitement": "NaN -> 0",
            "valeur_utilisee": 0.0,
            "nan_apres_traitement": train_remaining_nan,
        },
        {
            "dataset": "Test",
            "nan_avant_traitement": test_missing_count,
            "traitement": "NaN -> 0",
            "valeur_utilisee": 0.0,
            "nan_apres_traitement": test_remaining_nan,
        },
    ]
)

display(co2_reduction_report_df)

,dataset,nan_avant_traitement,traitement,valeur_utilisee,nan_apres_traitement
0,Train,35042,NaN -> 0,0.0,0
1,Test,8725,NaN -> 0,0.0,0


### 10.8 Imputation des valeurs manquantes rares

Conformément aux décisions prises au point 10.4, les variables
`wltp_test_mass_kg` et `engine_power_kw` présentent seulement quelques
valeurs manquantes résiduelles.

Aucune structure métier particulière n'a été mise en évidence pour ces NaN.

La stratégie retenue est donc une imputation par la médiane :

- la médiane est calculée exclusivement sur `X_train_processed` ;
- les NaN de `X_train_processed` sont remplacés par cette médiane ;
- la même médiane est ensuite appliquée aux NaN de `X_test_processed` ;
- aucune statistique n'est recalculée sur le jeu de test.

Cette étape ne concerne aucune autre variable.

In [18]:
# ---------------------------------------------------------------------
# 10.8 - Imputation des valeurs manquantes rares
# ---------------------------------------------------------------------

rare_missing_columns = [
    "wltp_test_mass_kg",
    "engine_power_kw",
]

rare_imputation_report = []


for column in rare_missing_columns:

    # Nombre de NaN avant traitement
    train_missing_count = int(
        X_train_processed[column]
        .isna()
        .sum()
    )

    test_missing_count = int(
        X_test_processed[column]
        .isna()
        .sum()
    )


    # Médiane calculée exclusivement sur X_train_processed
    median_train = (
        X_train_processed[column]
        .median()
    )

    if pd.isna(median_train):
        raise ValueError(
            f"Impossible de calculer la médiane de '{column}' "
            "à partir de X_train_processed."
        )


    # Imputation avec la même médiane pour Train et Test
    X_train_processed[column] = (
        X_train_processed[column]
        .fillna(median_train)
    )

    X_test_processed[column] = (
        X_test_processed[column]
        .fillna(median_train)
    )


    # Nombre de NaN après traitement
    train_remaining_nan = int(
        X_train_processed[column]
        .isna()
        .sum()
    )

    test_remaining_nan = int(
        X_test_processed[column]
        .isna()
        .sum()
    )


    # Rapport
    rare_imputation_report.extend(
        [
            {
                "variable": column,
                "dataset": "Train",
                "nan_avant_traitement": train_missing_count,
                "median_train": median_train,
                "nan_apres_traitement": train_remaining_nan,
            },
            {
                "variable": column,
                "dataset": "Test",
                "nan_avant_traitement": test_missing_count,
                "median_train": median_train,
                "nan_apres_traitement": test_remaining_nan,
            },
        ]
    )


rare_imputation_report_df = pd.DataFrame(
    rare_imputation_report
)

display(
    rare_imputation_report_df
)

,variable,dataset,nan_avant_traitement,median_train,nan_apres_traitement
0,wltp_test_mass_kg,Train,225,1609.0,0
1,wltp_test_mass_kg,Test,60,1609.0,0
2,engine_power_kw,Train,1,103.0,0
3,engine_power_kw,Test,0,103.0,0


### 10.9 Synthèse des traitements numériques appliqués

Les traitements définis au point 10.4 ont été appliqués aux étapes 10.6,
10.7 et 10.8.

Cette étape n'effectue aucune nouvelle imputation et ne modifie pas
`X_train_processed` ou `X_test_processed`.

Elle regroupe uniquement les résultats des traitements précédents dans un
tableau de synthèse permettant d'identifier, pour chaque variable :

- le jeu concerné (`Train` ou `Test`) ;
- la nature des valeurs manquantes ;
- le nombre de valeurs manquantes traitées ;
- le traitement réellement appliqué ;
- la valeur utilisée pour le remplacement ;
- l'origine de cette valeur.

Pour les imputations statistiques, la valeur utilisée est toujours une
médiane apprise exclusivement sur `X_train`.

Pour les absences structurelles ou informatives traitées par `0`, la valeur
provient de la règle métier définie au point 10.4.

In [19]:
# ---------------------------------------------------------------------
# 10.9 - Synthèse des traitements numériques appliqués
# ---------------------------------------------------------------------

imputation_summary = []


# ---------------------------------------------------------------------
# 1. Traitements conditionnels réalisés au point 10.6
# ---------------------------------------------------------------------

for _, row in conditional_imputation_report_df.iterrows():

    # NaN structurels - Train
    if row["train_structural_nan"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": "Train",
                "nature_nan": "Structurel",
                "nombre_nan_traites": int(
                    row["train_structural_nan"]
                ),
                "traitement": "Remplacement par 0",
                "valeur_utilisee": 0.0,
                "origine_valeur": "Règle métier",
            }
        )

    # NaN résiduels - Train
    if row["train_residual_nan"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": "Train",
                "nature_nan": "Résiduel",
                "nombre_nan_traites": int(
                    row["train_residual_nan"]
                ),
                "traitement": "Imputation par médiane",
                "valeur_utilisee": row["median_train"],
                "origine_valeur": "Médiane apprise sur X_train",
            }
        )

    # NaN structurels - Test
    if row["test_structural_nan"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": "Test",
                "nature_nan": "Structurel",
                "nombre_nan_traites": int(
                    row["test_structural_nan"]
                ),
                "traitement": "Remplacement par 0",
                "valeur_utilisee": 0.0,
                "origine_valeur": "Règle métier",
            }
        )

    # NaN résiduels - Test
    if row["test_residual_nan"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": "Test",
                "nature_nan": "Résiduel",
                "nombre_nan_traites": int(
                    row["test_residual_nan"]
                ),
                "traitement": "Imputation par médiane",
                "valeur_utilisee": row["median_train"],
                "origine_valeur": "Médiane apprise sur X_train",
            }
        )


# ---------------------------------------------------------------------
# 2. Traitement spécifique réalisé au point 10.7
# ---------------------------------------------------------------------

for _, row in co2_reduction_report_df.iterrows():

    if row["nan_avant_traitement"] > 0:
        imputation_summary.append(
            {
                "variable": "co2_reduction_wltp_g_km",
                "dataset": row["dataset"],
                "nature_nan": "Absence informative",
                "nombre_nan_traites": int(
                    row["nan_avant_traitement"]
                ),
                "traitement": "Remplacement par 0",
                "valeur_utilisee": 0.0,
                "origine_valeur": "Règle métier",
            }
        )


# ---------------------------------------------------------------------
# 3. Imputations des NaN rares réalisées au point 10.8
# ---------------------------------------------------------------------

for _, row in rare_imputation_report_df.iterrows():

    # Une ligne n'est ajoutée que lorsqu'une imputation
    # a réellement été nécessaire.
    if row["nan_avant_traitement"] > 0:
        imputation_summary.append(
            {
                "variable": row["variable"],
                "dataset": row["dataset"],
                "nature_nan": "Résiduel rare",
                "nombre_nan_traites": int(
                    row["nan_avant_traitement"]
                ),
                "traitement": "Imputation par médiane",
                "valeur_utilisee": row["median_train"],
                "origine_valeur": "Médiane apprise sur X_train",
            }
        )


# ---------------------------------------------------------------------
# 4. Construction du tableau de synthèse
# ---------------------------------------------------------------------

imputation_summary_df = pd.DataFrame(
    imputation_summary
)


# Ordre d'affichage des variables.
variable_order = [
    "engine_capacity_cm3",
    "electric_energy_consumption_wh_km",
    "electric_range_km",
    "fuel_consumption",
    "co2_reduction_wltp_g_km",
    "wltp_test_mass_kg",
    "engine_power_kw",
]

imputation_summary_df["variable"] = pd.Categorical(
    imputation_summary_df["variable"],
    categories=variable_order,
    ordered=True,
)


# Train affiché avant Test.
imputation_summary_df["dataset"] = pd.Categorical(
    imputation_summary_df["dataset"],
    categories=["Train", "Test"],
    ordered=True,
)


imputation_summary_df = (
    imputation_summary_df
    .sort_values(
        by=[
            "variable",
            "dataset",
            "nature_nan",
        ]
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 5. Affichage
# ---------------------------------------------------------------------

display(imputation_summary_df)

,variable,dataset,nature_nan,nombre_nan_traites,traitement,valeur_utilisee,origine_valeur
0,engine_capacity_cm3,Train,Structurel,11355,Remplacement par 0,0.0,Règle métier
1,engine_capacity_cm3,Test,Structurel,2840,Remplacement par 0,0.0,Règle métier
2,electric_energy_consumption_wh_km,Train,Résiduel,133,Imputation par médiane,167.0,Médiane apprise sur X_train
3,electric_energy_consumption_wh_km,Train,Structurel,63063,Remplacement par 0,0.0,Règle métier
4,electric_energy_consumption_wh_km,Test,Résiduel,33,Imputation par médiane,167.0,Médiane apprise sur X_train
5,electric_energy_consumption_wh_km,Test,Structurel,15733,Remplacement par 0,0.0,Règle métier
6,electric_range_km,Train,Résiduel,152,Imputation par médiane,394.0,Médiane apprise sur X_train
7,electric_range_km,Train,Structurel,63063,Remplacement par 0,0.0,Règle métier
8,electric_range_km,Test,Résiduel,39,Imputation par médiane,394.0,Médiane apprise sur X_train
9,electric_range_km,Test,Structurel,15733,Remplacement par 0,0.0,Règle métier


### 10.10 Validation finale des traitements appliqués

Les traitements définis au point 10.4 et appliqués aux points 10.5 à 10.8
sont maintenant contrôlés avant de poursuivre le preprocessing.

Cette étape n'effectue aucune nouvelle transformation.

Les vérifications portent sur :

1. **Les variables numériques traitées**

   Les variables numériques ayant fait l'objet d'un traitement ne doivent
   plus contenir de valeur manquante.

2. **Les indicateurs binaires**

   Les quatre indicateurs créés au point 10.5 doivent :

   - être présents dans `X_train_processed` et `X_test_processed` ;
   - ne contenir aucune valeur manquante ;
   - contenir uniquement les valeurs `0` et `1`.

3. **Les autres variables**

   Les éventuelles valeurs manquantes encore présentes dans les autres
   variables sont identifiées, sans être traitées à cette étape.

L'objectif est de valider les traitements numériques réalisés avant de
poursuivre avec le traitement des variables restantes et la construction
du preprocessing final.

In [20]:
# ---------------------------------------------------------------------
# 10.10 - Validation finale des traitements appliqués
# ---------------------------------------------------------------------

# Variables numériques traitées aux points 10.6, 10.7 et 10.8.
treated_numeric_columns = [
    "engine_capacity_cm3",
    "electric_energy_consumption_wh_km",
    "electric_range_km",
    "fuel_consumption",
    "co2_reduction_wltp_g_km",
    "wltp_test_mass_kg",
    "engine_power_kw",
]


# Indicateurs binaires créés au point 10.5.
indicator_columns = [
    "has_electric_energy_consumption_wh_km",
    "has_electric_range_km",
    "has_fuel_consumption",
    "has_co2_reduction_wltp_g_km",
]


# ---------------------------------------------------------------------
# 1. Vérification des variables numériques traitées
# ---------------------------------------------------------------------

numeric_validation = []

for column in treated_numeric_columns:

    train_nan = int(
        X_train_processed[column]
        .isna()
        .sum()
    )

    test_nan = int(
        X_test_processed[column]
        .isna()
        .sum()
    )

    numeric_validation.append(
        {
            "variable": column,
            "nan_train": train_nan,
            "nan_test": test_nan,
            "statut": (
                "OK"
                if train_nan == 0 and test_nan == 0
                else "À vérifier"
            ),
        }
    )


numeric_validation_df = pd.DataFrame(
    numeric_validation
)

print("VARIABLES NUMÉRIQUES TRAITÉES")
display(numeric_validation_df)


# ---------------------------------------------------------------------
# 2. Vérification des indicateurs binaires
# ---------------------------------------------------------------------

indicator_validation = []

for column in indicator_columns:

    train_exists = column in X_train_processed.columns
    test_exists = column in X_test_processed.columns

    if train_exists and test_exists:

        train_nan = int(
            X_train_processed[column]
            .isna()
            .sum()
        )

        test_nan = int(
            X_test_processed[column]
            .isna()
            .sum()
        )

        train_values = set(
            X_train_processed[column]
            .dropna()
            .unique()
        )

        test_values = set(
            X_test_processed[column]
            .dropna()
            .unique()
        )

        binary_values_valid = (
            train_values.issubset({0, 1})
            and test_values.issubset({0, 1})
        )

        valid = (
            train_nan == 0
            and test_nan == 0
            and binary_values_valid
        )

    else:

        train_nan = None
        test_nan = None
        binary_values_valid = False
        valid = False


    indicator_validation.append(
        {
            "indicateur": column,
            "present_train": train_exists,
            "present_test": test_exists,
            "nan_train": train_nan,
            "nan_test": test_nan,
            "valeurs_binaires_valides": binary_values_valid,
            "statut": "OK" if valid else "À vérifier",
        }
    )


indicator_validation_df = pd.DataFrame(
    indicator_validation
)

print("\nINDICATEURS BINAIRES")
display(indicator_validation_df)


# ---------------------------------------------------------------------
# 3. Recherche des NaN encore présents dans les autres variables
# ---------------------------------------------------------------------

remaining_train_nan = (
    X_train_processed
    .isna()
    .sum()
)

remaining_train_nan = remaining_train_nan[
    remaining_train_nan > 0
]


remaining_test_nan = (
    X_test_processed
    .isna()
    .sum()
)

remaining_test_nan = remaining_test_nan[
    remaining_test_nan > 0
]


remaining_nan_columns = sorted(
    set(remaining_train_nan.index)
    | set(remaining_test_nan.index)
)


remaining_nan_report = []

for column in remaining_nan_columns:

    remaining_nan_report.append(
        {
            "variable": column,
            "nan_train": int(
                remaining_train_nan.get(column, 0)
            ),
            "nan_test": int(
                remaining_test_nan.get(column, 0)
            ),
        }
    )


remaining_nan_report_df = pd.DataFrame(
    remaining_nan_report
)


print("\nVALEURS MANQUANTES ENCORE PRÉSENTES")

if remaining_nan_report_df.empty:
    print(
        "Aucune valeur manquante restante "
        "dans X_train_processed et X_test_processed."
    )
else:
    display(remaining_nan_report_df)


# ---------------------------------------------------------------------
# 4. Validation des traitements numériques
# ---------------------------------------------------------------------

numeric_treatment_valid = (
    numeric_validation_df["statut"]
    .eq("OK")
    .all()
)

indicators_valid = (
    indicator_validation_df["statut"]
    .eq("OK")
    .all()
)


if numeric_treatment_valid and indicators_valid:
    print(
        "\n✅ Les traitements numériques et les indicateurs "
        "binaires sont validés."
    )
else:
    raise ValueError(
        "La validation des traitements numériques "
        "ou des indicateurs binaires a échoué."
    )

VARIABLES NUMÉRIQUES TRAITÉES


,variable,nan_train,nan_test,statut
0,engine_capacity_cm3,0,0,OK
1,electric_energy_consumption_wh_km,0,0,OK
2,electric_range_km,0,0,OK
3,fuel_consumption,0,0,OK
4,co2_reduction_wltp_g_km,0,0,OK
5,wltp_test_mass_kg,0,0,OK
6,engine_power_kw,0,0,OK



INDICATEURS BINAIRES


,indicateur,present_train,present_test,nan_train,nan_test,valeurs_binaires_valides,statut
0,has_electric_energy_consumption_wh_km,True,True,0,0,True,OK
1,has_electric_range_km,True,True,0,0,True,OK
2,has_fuel_consumption,True,True,0,0,True,OK
3,has_co2_reduction_wltp_g_km,True,True,0,0,True,OK



VALEURS MANQUANTES ENCORE PRÉSENTES


,variable,nan_train,nan_test
0,manufacturer_make,3,1



✅ Les traitements numériques et les indicateurs binaires sont validés.


### 10.11 Traitement des valeurs manquantes de `manufacturer_make`

La validation réalisée au point 10.10 montre que les traitements numériques
sont terminés, mais que quelques valeurs manquantes subsistent dans la variable
catégorielle nominale `manufacturer_make` :

- 3 observations dans `X_train_processed` ;
- 1 observation dans `X_test_processed`.

Compte tenu du très faible nombre de valeurs manquantes et de la nature
nominale de cette variable, la stratégie retenue est une imputation par
la **modalité la plus fréquente**.

Afin d'éviter toute fuite d'information :

- la modalité la plus fréquente est déterminée exclusivement à partir de
  `X_train_processed` ;
- les valeurs manquantes de `X_train_processed` sont remplacées par cette
  modalité ;
- la même modalité apprise sur `X_train_processed` est ensuite utilisée
  pour `X_test_processed`.

Aucune statistique n'est calculée à partir de `X_test_processed`.

In [21]:
# ---------------------------------------------------------------------
# 10.11 - Traitement des valeurs manquantes de manufacturer_make
# ---------------------------------------------------------------------

column = "manufacturer_make"


# ---------------------------------------------------------------------
# 1. Nombre de NaN avant traitement
# ---------------------------------------------------------------------

train_missing_count = int(
    X_train_processed[column]
    .isna()
    .sum()
)

test_missing_count = int(
    X_test_processed[column]
    .isna()
    .sum()
)


# ---------------------------------------------------------------------
# 2. Apprentissage de la modalité la plus fréquente sur X_train
# ---------------------------------------------------------------------

train_mode = (
    X_train_processed[column]
    .mode(dropna=True)
)

if train_mode.empty:
    raise ValueError(
        f"Impossible de déterminer la modalité la plus fréquente "
        f"de '{column}' à partir de X_train_processed."
    )

most_frequent_train = train_mode.iloc[0]


# ---------------------------------------------------------------------
# 3. Imputation de Train et Test
#
# La valeur utilisée dans les deux datasets est exclusivement
# celle apprise sur X_train_processed.
# ---------------------------------------------------------------------

X_train_processed[column] = (
    X_train_processed[column]
    .fillna(most_frequent_train)
)

X_test_processed[column] = (
    X_test_processed[column]
    .fillna(most_frequent_train)
)


# ---------------------------------------------------------------------
# 4. Nombre de NaN après traitement
# ---------------------------------------------------------------------

train_remaining_nan = int(
    X_train_processed[column]
    .isna()
    .sum()
)

test_remaining_nan = int(
    X_test_processed[column]
    .isna()
    .sum()
)


# ---------------------------------------------------------------------
# 5. Rapport du traitement
# ---------------------------------------------------------------------

manufacturer_make_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train",
            "nan_avant_traitement": train_missing_count,
            "modalite_apprise_sur_train": most_frequent_train,
            "nan_apres_traitement": train_remaining_nan,
        },
        {
            "dataset": "Test",
            "nan_avant_traitement": test_missing_count,
            "modalite_apprise_sur_train": most_frequent_train,
            "nan_apres_traitement": test_remaining_nan,
        },
    ]
)

display(manufacturer_make_report_df)

,dataset,nan_avant_traitement,modalite_apprise_sur_train,nan_apres_traitement
0,Train,3,VOLKSWAGEN VW,0
1,Test,1,VOLKSWAGEN VW,0


### 10.12 Validation globale des valeurs manquantes

Les traitements des valeurs manquantes numériques et catégorielles ayant été
appliqués, une vérification globale est réalisée avant de poursuivre le
preprocessing.

Cette étape n'effectue aucune transformation supplémentaire.

Elle vérifie que :

- `X_train_processed` ne contient plus aucune valeur manquante ;
- `X_test_processed` ne contient plus aucune valeur manquante.

Cette validation clôture le traitement des valeurs manquantes avant les
étapes suivantes d'encodage des variables catégorielles et de mise à
l'échelle des variables numériques.

In [22]:
# ---------------------------------------------------------------------
# 10.12 - Validation globale des valeurs manquantes
# ---------------------------------------------------------------------

# Nombre total de NaN restants.
train_total_nan = int(
    X_train_processed
    .isna()
    .sum()
    .sum()
)

test_total_nan = int(
    X_test_processed
    .isna()
    .sum()
    .sum()
)


# ---------------------------------------------------------------------
# Affichage du contrôle
# ---------------------------------------------------------------------

missing_validation_df = pd.DataFrame(
    [
        {
            "dataset": "Train",
            "nombre_total_nan": train_total_nan,
            "statut": "OK" if train_total_nan == 0 else "À vérifier",
        },
        {
            "dataset": "Test",
            "nombre_total_nan": test_total_nan,
            "statut": "OK" if test_total_nan == 0 else "À vérifier",
        },
    ]
)

display(missing_validation_df)


# ---------------------------------------------------------------------
# Validation
# ---------------------------------------------------------------------

if train_total_nan != 0 or test_total_nan != 0:

    raise ValueError(
        "Des valeurs manquantes subsistent dans "
        "X_train_processed ou X_test_processed."
    )


print(
    "✅ Le traitement des valeurs manquantes est terminé : "
    "aucun NaN ne subsiste dans X_train_processed "
    "et X_test_processed."
)

,dataset,nombre_total_nan,statut
0,Train,0,OK
1,Test,0,OK


✅ Le traitement des valeurs manquantes est terminé : aucun NaN ne subsiste dans X_train_processed et X_test_processed.


### 10.13 Stratégie d'encodage des variables catégorielles nominales

Les valeurs manquantes ayant été entièrement traitées et validées, l'étape
suivante consiste à préparer l'encodage des variables catégorielles nominales.

Les variables catégorielles identifiées précédemment sont :

- `manufacturer_make` ;
- `vehicle_category_type` ;
- `fuel_type` ;
- `fuel_mode`.

Une **stratégie d'encodage hybride** est retenue afin d'adapter le traitement
à la cardinalité des différentes variables.

#### Variables nominales à faible cardinalité

Les variables :

- `vehicle_category_type` ;
- `fuel_type` ;
- `fuel_mode`

seront traitées par **One-Hot Encoding**.

Chaque modalité sera ainsi représentée par une variable binaire indépendante.

L'encodeur sera ajusté exclusivement sur `X_train_processed`, puis utilisé
pour transformer `X_train_processed` et `X_test_processed`.

Il devra également permettre la gestion d'éventuelles modalités présentes
dans les données à transformer mais absentes des données d'entraînement.

#### Variable `manufacturer_make`

La variable `manufacturer_make` présente une cardinalité plus importante.

Afin d'éviter la création d'un nombre important de variables binaires,
un **Frequency Encoding** est retenu pour cette variable.

Chaque constructeur sera remplacé par sa fréquence d'apparition calculée
exclusivement à partir de `X_train_processed`.

La table de fréquences apprise sur Train sera ensuite utilisée pour transformer :

- `X_train_processed` ;
- `X_test_processed`.

Une modalité rencontrée dans Test mais absente de Train sera considérée comme
inconnue et recevra une fréquence de `0`.

Cette stratégie permet de conserver une représentation numérique compacte de
`manufacturer_make` sans utiliser les informations de `X_test_processed`
pendant l'apprentissage de l'encodage.

#### Contrôle préalable

Avant l'application des encodeurs, la cardinalité des variables catégorielles
et la présence éventuelle de modalités de Test absentes de Train sont mesurées
dynamiquement.

Aucune cardinalité n'est considérée comme fixe : les résultats dépendent des
données réellement utilisées lors de l'exécution.

In [23]:
# ---------------------------------------------------------------------
# 10.13 - Contrôle préalable à l'encodage des variables catégorielles
# ---------------------------------------------------------------------

categorical_columns = [
    "manufacturer_make",
    "vehicle_category_type",
    "fuel_type",
    "fuel_mode",
]


# ---------------------------------------------------------------------
# 1. Contrôle de présence des variables
# ---------------------------------------------------------------------

missing_train_columns = [
    column
    for column in categorical_columns
    if column not in X_train_processed.columns
]

missing_test_columns = [
    column
    for column in categorical_columns
    if column not in X_test_processed.columns
]

if missing_train_columns or missing_test_columns:
    raise ValueError(
        "Certaines variables catégorielles sont absentes des datasets.\n"
        f"Absentes de X_train_processed : {missing_train_columns}\n"
        f"Absentes de X_test_processed : {missing_test_columns}"
    )


# ---------------------------------------------------------------------
# 2. Cardinalité observée dans Train et Test
# ---------------------------------------------------------------------

categorical_cardinality_report = []

for column in categorical_columns:

    train_categories = set(
        X_train_processed[column]
        .dropna()
        .unique()
    )

    test_categories = set(
        X_test_processed[column]
        .dropna()
        .unique()
    )

    unseen_test_categories = (
        test_categories - train_categories
    )

    categorical_cardinality_report.append(
        {
            "variable": column,
            "modalites_train": len(train_categories),
            "modalites_test": len(test_categories),
            "modalites_test_absentes_train": len(
                unseen_test_categories
            ),
        }
    )


categorical_cardinality_df = pd.DataFrame(
    categorical_cardinality_report
)

print("CARDINALITÉ DES VARIABLES CATÉGORIELLES")
display(categorical_cardinality_df)


# ---------------------------------------------------------------------
# 3. Identification des modalités présentes dans Test
#    mais absentes de Train
# ---------------------------------------------------------------------

unseen_categories_report = []

for column in categorical_columns:

    train_categories = set(
        X_train_processed[column]
        .dropna()
        .unique()
    )

    test_categories = set(
        X_test_processed[column]
        .dropna()
        .unique()
    )

    unseen_categories = sorted(
        test_categories - train_categories
    )

    if unseen_categories:

        for category in unseen_categories:
            unseen_categories_report.append(
                {
                    "variable": column,
                    "modalite_absente_train": category,
                }
            )


unseen_categories_df = pd.DataFrame(
    unseen_categories_report
)


# ---------------------------------------------------------------------
# 4. Affichage des éventuelles catégories inconnues
# ---------------------------------------------------------------------

print("\nMODALITÉS PRÉSENTES DANS TEST MAIS ABSENTES DE TRAIN")

if unseen_categories_df.empty:

    print(
        "Aucune modalité de X_test_processed "
        "n'est absente de X_train_processed."
    )

else:

    display(unseen_categories_df)


# ---------------------------------------------------------------------
# 5. Conclusion du contrôle
# ---------------------------------------------------------------------

print(
    "\n✅ Contrôle préalable à l'encodage terminé."
)

CARDINALITÉ DES VARIABLES CATÉGORIELLES


,variable,modalites_train,modalites_test,modalites_test_absentes_train
0,manufacturer_make,93,81,5
1,vehicle_category_type,2,1,0
2,fuel_type,9,9,0
3,fuel_mode,6,6,0



MODALITÉS PRÉSENTES DANS TEST MAIS ABSENTES DE TRAIN


,variable,modalite_absente_train
0,manufacturer_make,HYUNDAI MOTOR (ROK)
1,manufacturer_make,LADA
2,manufacturer_make,MERCEDES AMG
3,manufacturer_make,RIMOR
4,manufacturer_make,ROLLS ROYCE



✅ Contrôle préalable à l'encodage terminé.


### 10.14 Frequency Encoding de `manufacturer_make`

Conformément à la stratégie d'encodage hybride définie au point 10.13,
`manufacturer_make` est traitée par **Frequency Encoding**.

La fréquence de chaque constructeur est calculée exclusivement à partir de
`X_train_processed`.

La table de fréquences ainsi apprise est ensuite utilisée pour transformer :

- `X_train_processed` ;
- `X_test_processed`.

La variable catégorielle `manufacturer_make` est remplacée par une nouvelle
variable numérique :

`manufacturer_make_frequency`

Cette variable représente la fréquence d'apparition du constructeur dans les
données d'entraînement.

Lorsqu'un constructeur présent dans `X_test_processed` n'existe pas dans
`X_train_processed`, aucune fréquence ne peut être apprise pour cette modalité.
La valeur `0` lui est alors attribuée.

Cette méthode garantit que les informations de `X_test_processed` ne sont pas
utilisées pendant l'apprentissage de l'encodage.

In [24]:
# ---------------------------------------------------------------------
# 10.14 - Frequency Encoding de manufacturer_make
# ---------------------------------------------------------------------

source_column = "manufacturer_make"
encoded_column = "manufacturer_make_frequency"


# ---------------------------------------------------------------------
# 1. Apprentissage des fréquences exclusivement sur X_train_processed
# ---------------------------------------------------------------------

manufacturer_frequency_map = (
    X_train_processed[source_column]
    .value_counts(normalize=True)
)


# ---------------------------------------------------------------------
# 2. Transformation de X_train_processed
# ---------------------------------------------------------------------

X_train_processed[encoded_column] = (
    X_train_processed[source_column]
    .map(manufacturer_frequency_map)
    .fillna(0.0)
)


# ---------------------------------------------------------------------
# 3. Transformation de X_test_processed
#
# Une modalité inconnue de Train ne possède aucune fréquence apprise.
# Elle reçoit donc la valeur 0.
# ---------------------------------------------------------------------

X_test_processed[encoded_column] = (
    X_test_processed[source_column]
    .map(manufacturer_frequency_map)
    .fillna(0.0)
)


# ---------------------------------------------------------------------
# 4. Identification des modalités inconnues rencontrées dans Test
# ---------------------------------------------------------------------

unknown_test_mask = (
    ~X_test_processed[source_column]
    .isin(manufacturer_frequency_map.index)
)

unknown_test_rows = int(
    unknown_test_mask.sum()
)

unknown_test_categories = sorted(
    X_test_processed.loc[
        unknown_test_mask,
        source_column,
    ]
    .unique()
    .tolist()
)


# ---------------------------------------------------------------------
# 5. Vérification de l'encodage
# ---------------------------------------------------------------------

train_encoded_nan = int(
    X_train_processed[encoded_column]
    .isna()
    .sum()
)

test_encoded_nan = int(
    X_test_processed[encoded_column]
    .isna()
    .sum()
)


frequency_encoding_report_df = pd.DataFrame(
    [
        {
            "dataset": "Train",
            "nan_apres_encodage": train_encoded_nan,
            "modalites_inconnues": 0,
            "observations_modalites_inconnues": 0,
        },
        {
            "dataset": "Test",
            "nan_apres_encodage": test_encoded_nan,
            "modalites_inconnues": len(
                unknown_test_categories
            ),
            "observations_modalites_inconnues": unknown_test_rows,
        },
    ]
)

display(frequency_encoding_report_df)


# ---------------------------------------------------------------------
# 6. Affichage des modalités inconnues de Test
# ---------------------------------------------------------------------

if unknown_test_categories:

    print(
        "\nModalités de manufacturer_make présentes dans Test "
        "mais absentes de Train :"
    )

    for category in unknown_test_categories:
        print(f" - {category}")

    print(
        "\nCes modalités ont reçu "
        "manufacturer_make_frequency = 0."
    )

else:

    print(
        "\nAucune modalité inconnue de manufacturer_make "
        "n'a été rencontrée dans Test."
    )


# ---------------------------------------------------------------------
# 7. Suppression de la variable catégorielle originale
# ---------------------------------------------------------------------

X_train_processed.drop(
    columns=[source_column],
    inplace=True,
)

X_test_processed.drop(
    columns=[source_column],
    inplace=True,
)


print(
    "\n✅ Frequency Encoding de manufacturer_make terminé."
)

,dataset,nan_apres_encodage,modalites_inconnues,observations_modalites_inconnues
0,Train,0,0,0
1,Test,0,5,6



Modalités de manufacturer_make présentes dans Test mais absentes de Train :
 - HYUNDAI MOTOR (ROK)
 - LADA
 - MERCEDES AMG
 - RIMOR
 - ROLLS ROYCE

Ces modalités ont reçu manufacturer_make_frequency = 0.

✅ Frequency Encoding de manufacturer_make terminé.


### 10.15 One-Hot Encoding des variables catégorielles à faible cardinalité

Conformément à la stratégie d'encodage hybride définie au point 10.13,
les variables catégorielles suivantes sont traitées par **One-Hot Encoding** :

- `vehicle_category_type` ;
- `fuel_type` ;
- `fuel_mode`.

L'encodeur est ajusté exclusivement sur `X_train_processed`.

Les catégories apprises sur le jeu d'entraînement sont ensuite utilisées
pour transformer :

- `X_train_processed` ;
- `X_test_processed`.

La gestion des éventuelles catégories inconnues est assurée par
`handle_unknown="ignore"` afin qu'une modalité absente de Train ne provoque
pas d'erreur lors de la transformation de Test ou de futures données.

Les colonnes catégorielles originales sont ensuite remplacées par les
variables binaires produites par l'encodage.

La variable `manufacturer_make` n'est pas concernée par cette étape puisqu'elle
a déjà été transformée par Frequency Encoding au point 10.14.

In [25]:
# ---------------------------------------------------------------------
# 10.15 - One-Hot Encoding des variables catégorielles
#         à faible cardinalité
# ---------------------------------------------------------------------

from sklearn.preprocessing import OneHotEncoder


# Variables concernées uniquement par le One-Hot Encoding.
onehot_columns = [
    "vehicle_category_type",
    "fuel_type",
    "fuel_mode",
]


# ---------------------------------------------------------------------
# 1. Création et apprentissage de l'encodeur sur Train uniquement
# ---------------------------------------------------------------------

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype="int8",
)

onehot_encoder.fit(
    X_train_processed[onehot_columns]
)


# ---------------------------------------------------------------------
# 2. Transformation de Train et Test
# ---------------------------------------------------------------------

X_train_onehot_array = onehot_encoder.transform(
    X_train_processed[onehot_columns]
)

X_test_onehot_array = onehot_encoder.transform(
    X_test_processed[onehot_columns]
)


# ---------------------------------------------------------------------
# 3. Récupération dynamique des noms des nouvelles variables
# ---------------------------------------------------------------------

onehot_feature_names = (
    onehot_encoder
    .get_feature_names_out(onehot_columns)
)


# ---------------------------------------------------------------------
# 4. Conversion des résultats en DataFrames
# ---------------------------------------------------------------------

X_train_onehot = pd.DataFrame(
    X_train_onehot_array,
    columns=onehot_feature_names,
    index=X_train_processed.index,
)

X_test_onehot = pd.DataFrame(
    X_test_onehot_array,
    columns=onehot_feature_names,
    index=X_test_processed.index,
)


# ---------------------------------------------------------------------
# 5. Suppression des variables catégorielles originales
# ---------------------------------------------------------------------

X_train_processed.drop(
    columns=onehot_columns,
    inplace=True,
)

X_test_processed.drop(
    columns=onehot_columns,
    inplace=True,
)


# ---------------------------------------------------------------------
# 6. Ajout des variables One-Hot encodées
# ---------------------------------------------------------------------

X_train_processed = pd.concat(
    [
        X_train_processed,
        X_train_onehot,
    ],
    axis=1,
)

X_test_processed = pd.concat(
    [
        X_test_processed,
        X_test_onehot,
    ],
    axis=1,
)


# ---------------------------------------------------------------------
# 7. Vérification de la cohérence des colonnes Train / Test
# ---------------------------------------------------------------------

same_columns = (
    X_train_processed.columns.tolist()
    == X_test_processed.columns.tolist()
)

if not same_columns:
    raise ValueError(
        "Les colonnes de X_train_processed et X_test_processed "
        "ne sont pas identiques après One-Hot Encoding."
    )


# ---------------------------------------------------------------------
# 8. Rapport du traitement
# ---------------------------------------------------------------------

onehot_report_df = pd.DataFrame(
    [
        {
            "variable_source": column,
            "modalites_apprises_train": len(
                onehot_encoder.categories_[index]
            ),
        }
        for index, column in enumerate(onehot_columns)
    ]
)

display(onehot_report_df)

print(
    f"\nNombre total de variables créées par One-Hot Encoding : "
    f"{len(onehot_feature_names)}"
)

print(
    "\n✅ One-Hot Encoding terminé : "
    "Train et Test possèdent exactement les mêmes colonnes."
)

,variable_source,modalites_apprises_train
0,vehicle_category_type,2
1,fuel_type,9
2,fuel_mode,6



Nombre total de variables créées par One-Hot Encoding : 17

✅ One-Hot Encoding terminé : Train et Test possèdent exactement les mêmes colonnes.


### 10.16 Standardisation des variables numériques continues

Après le traitement des valeurs manquantes et l'encodage des variables
catégorielles, les variables numériques continues sont mises à l'échelle
à l'aide d'un **StandardScaler**.

La standardisation transforme chaque variable à partir de la moyenne et de
l'écart-type appris sur les données d'entraînement.

Afin d'éviter toute fuite d'information :

- le `StandardScaler` est ajusté exclusivement sur `X_train_processed` ;
- les paramètres appris sur Train sont utilisés pour transformer
  `X_train_processed` ;
- ces mêmes paramètres sont ensuite utilisés pour transformer
  `X_test_processed`.

Les variables numériques continues initiales ainsi que
`manufacturer_make_frequency`, issue du Frequency Encoding, sont concernées
par cette transformation.

Les variables binaires ne sont pas standardisées :

- les indicateurs `has_*` créés lors du traitement des valeurs manquantes ;
- les variables binaires produites par le One-Hot Encoding.

Elles conservent donc directement leurs valeurs `0` et `1`.

La liste des variables à standardiser est déterminée dynamiquement à partir
des données afin de rester compatible avec le passage ultérieur au pipeline
FULL.

In [26]:
# ---------------------------------------------------------------------
# 10.16 - Standardisation des variables numériques continues
# ---------------------------------------------------------------------

from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------------------
# 1. Identification des variables binaires à ne pas standardiser
# ---------------------------------------------------------------------

binary_indicator_columns = [
    "has_electric_energy_consumption_wh_km",
    "has_electric_range_km",
    "has_fuel_consumption",
    "has_co2_reduction_wltp_g_km",
]

# Variables créées par le One-Hot Encoding au point 10.15.
onehot_encoded_columns = list(
    onehot_feature_names
)

excluded_from_scaling = (
    binary_indicator_columns
    + onehot_encoded_columns
)


# ---------------------------------------------------------------------
# 2. Identification dynamique des variables numériques continues
# ---------------------------------------------------------------------

numeric_columns = (
    X_train_processed
    .select_dtypes(include="number")
    .columns
    .tolist()
)

columns_to_scale = [
    column
    for column in numeric_columns
    if column not in excluded_from_scaling
]


# ---------------------------------------------------------------------
# 3. Contrôle de cohérence Train / Test
# ---------------------------------------------------------------------

missing_in_test = [
    column
    for column in columns_to_scale
    if column not in X_test_processed.columns
]

if missing_in_test:
    raise ValueError(
        "Certaines variables numériques de Train sont absentes de Test : "
        f"{missing_in_test}"
    )


# ---------------------------------------------------------------------
# 4. Apprentissage du StandardScaler exclusivement sur Train
# ---------------------------------------------------------------------

standard_scaler = StandardScaler()

standard_scaler.fit(
    X_train_processed[columns_to_scale]
)


# ---------------------------------------------------------------------
# 5. Transformation de Train et Test
# ---------------------------------------------------------------------

X_train_processed[columns_to_scale] = (
    standard_scaler.transform(
        X_train_processed[columns_to_scale]
    )
)

X_test_processed[columns_to_scale] = (
    standard_scaler.transform(
        X_test_processed[columns_to_scale]
    )
)


# ---------------------------------------------------------------------
# 6. Rapport de standardisation
# ---------------------------------------------------------------------

scaling_report_df = pd.DataFrame(
    {
        "variable": columns_to_scale,
        "moyenne_apprise_train": standard_scaler.mean_,
        "ecart_type_appris_train": standard_scaler.scale_,
    }
)

display(scaling_report_df)


print(
    f"\nNombre de variables standardisées : "
    f"{len(columns_to_scale)}"
)

print(
    f"Nombre de variables binaires exclues du scaling : "
    f"{len(excluded_from_scaling)}"
)

print(
    "\n✅ StandardScaler appris exclusivement sur Train "
    "et appliqué à Train et Test."
)

,variable,moyenne_apprise_train,ecart_type_appris_train
0,mass_running_order_kg,1566.452712,359.187662
1,wltp_test_mass_kg,1686.629600,379.390468
2,engine_capacity_cm3,1352.332175,747.314823
3,engine_power_kw,117.807812,62.547350
4,electric_energy_consumption_wh_km,37.027475,73.087651
5,co2_reduction_wltp_g_km,0.842596,0.844135
6,fuel_consumption,4.717263,2.524487
7,electric_range_km,69.009563,160.329387
8,registration_month_sin,0.036490,0.711809
9,registration_month_cos,0.006502,0.701395



Nombre de variables standardisées : 11
Nombre de variables binaires exclues du scaling : 21

✅ StandardScaler appris exclusivement sur Train et appliqué à Train et Test.


### 10.17 Validation finale du preprocessing

Les différentes étapes de preprocessing ayant été appliquées, une validation
finale est réalisée avant de considérer les jeux de données comme prêts pour
la modélisation.

Cette étape n'effectue aucune nouvelle transformation.

Elle vérifie que :

- `X_train_processed` et `X_test_processed` ne contiennent plus de valeurs
  manquantes ;
- aucune valeur numérique infinie (`+inf` ou `-inf`) n'est présente ;
- aucune variable de type `object` ne subsiste après les encodages ;
- Train et Test possèdent exactement les mêmes variables explicatives ;
- le nombre d'observations de `X_train_processed` correspond à celui de
  `y_train` ;
- le nombre d'observations de `X_test_processed` correspond à celui de
  `y_test`.

Cette validation permet de confirmer que les données issues du preprocessing
sont cohérentes et prêtes à être utilisées par les modèles de Machine Learning.

In [27]:
# ---------------------------------------------------------------------
# 10.17 - Validation finale du preprocessing
# ---------------------------------------------------------------------

import numpy as np


# ---------------------------------------------------------------------
# 1. Valeurs manquantes
# ---------------------------------------------------------------------

train_nan_count = int(
    X_train_processed.isna().sum().sum()
)

test_nan_count = int(
    X_test_processed.isna().sum().sum()
)


# ---------------------------------------------------------------------
# 2. Valeurs infinies
# ---------------------------------------------------------------------

train_numeric = X_train_processed.select_dtypes(
    include="number"
)

test_numeric = X_test_processed.select_dtypes(
    include="number"
)

train_inf_count = int(
    np.isinf(train_numeric.to_numpy()).sum()
)

test_inf_count = int(
    np.isinf(test_numeric.to_numpy()).sum()
)


# ---------------------------------------------------------------------
# 3. Variables non numériques restantes
# ---------------------------------------------------------------------

train_non_numeric_columns = (
    X_train_processed
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

test_non_numeric_columns = (
    X_test_processed
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)


# ---------------------------------------------------------------------
# 4. Cohérence des variables entre Train et Test
# ---------------------------------------------------------------------

same_columns = (
    X_train_processed.columns.tolist()
    == X_test_processed.columns.tolist()
)


# ---------------------------------------------------------------------
# 5. Cohérence entre X et y
# ---------------------------------------------------------------------

train_rows_match = (
    len(X_train_processed) == len(y_train)
)

test_rows_match = (
    len(X_test_processed) == len(y_test)
)


# ---------------------------------------------------------------------
# 6. Rapport final
# ---------------------------------------------------------------------

validation_report_df = pd.DataFrame(
    [
        {
            "controle": "Valeurs manquantes",
            "train": train_nan_count,
            "test": test_nan_count,
            "statut": (
                "OK"
                if train_nan_count == 0
                and test_nan_count == 0
                else "À vérifier"
            ),
        },
        {
            "controle": "Valeurs infinies",
            "train": train_inf_count,
            "test": test_inf_count,
            "statut": (
                "OK"
                if train_inf_count == 0
                and test_inf_count == 0
                else "À vérifier"
            ),
        },
        {
            "controle": "Variables non numériques",
            "train": len(train_non_numeric_columns),
            "test": len(test_non_numeric_columns),
            "statut": (
                "OK"
                if not train_non_numeric_columns
                and not test_non_numeric_columns
                else "À vérifier"
            ),
        },
        {
            "controle": "Même structure de variables",
            "train": same_columns,
            "test": same_columns,
            "statut": (
                "OK"
                if same_columns
                else "À vérifier"
            ),
        },
        {
            "controle": "Cohérence X / y",
            "train": train_rows_match,
            "test": test_rows_match,
            "statut": (
                "OK"
                if train_rows_match
                and test_rows_match
                else "À vérifier"
            ),
        },
    ]
)

display(validation_report_df)


# ---------------------------------------------------------------------
# 7. Dimensions finales
# ---------------------------------------------------------------------

dimensions_report_df = pd.DataFrame(
    [
        {
            "dataset": "X_train_processed",
            "observations": X_train_processed.shape[0],
            "variables": X_train_processed.shape[1],
        },
        {
            "dataset": "X_test_processed",
            "observations": X_test_processed.shape[0],
            "variables": X_test_processed.shape[1],
        },
    ]
)

print("\nDIMENSIONS FINALES")
display(dimensions_report_df)


# ---------------------------------------------------------------------
# 8. Validation stricte
# ---------------------------------------------------------------------

all_checks_valid = (
    train_nan_count == 0
    and test_nan_count == 0
    and train_inf_count == 0
    and test_inf_count == 0
    and not train_non_numeric_columns
    and not test_non_numeric_columns
    and same_columns
    and train_rows_match
    and test_rows_match
)

if not all_checks_valid:
    raise ValueError(
        "La validation finale du preprocessing a échoué. "
        "Consulter le tableau de contrôle."
    )


print(
    "\n✅ Validation finale réussie : "
    "X_train_processed et X_test_processed sont prêts "
    "pour la modélisation."
)

,controle,train,test,statut
0,Valeurs manquantes,0,0,OK
1,Valeurs infinies,0,0,OK
2,Variables non numériques,0,0,OK
3,Même structure de variables,True,True,OK
4,Cohérence X / y,True,True,OK



DIMENSIONS FINALES


,dataset,observations,variables
0,X_train_processed,80000,32
1,X_test_processed,20000,32



✅ Validation finale réussie : X_train_processed et X_test_processed sont prêts pour la modélisation.


### 10.18 Sauvegarde des jeux de données prétraités

Le preprocessing ayant été entièrement appliqué et validé, les jeux de données
obtenus sont maintenant sauvegardés afin d'être réutilisés dans les étapes
suivantes du projet, notamment pour l'entraînement et l'évaluation des modèles.

Les quatre jeux issus de la séparation Train / Test sont sauvegardés :

- `X_train_processed` : variables explicatives d'entraînement prétraitées ;
- `X_test_processed` : variables explicatives de test prétraitées ;
- `y_train` : cible associée au jeu d'entraînement ;
- `y_test` : cible associée au jeu de test.

Les fichiers sont enregistrés dans le répertoire prévu pour les données
prétraitées du projet.

Cette sauvegarde permet de séparer clairement la responsabilité du présent
notebook — préparation des données pour le Machine Learning — de celle des
prochaines étapes consacrées à la modélisation.

Les dimensions et la structure des données sauvegardées correspondent aux
objets validés au point 10.17.

In [28]:
# ---------------------------------------------------------------------
# 10.18 - Sauvegarde des jeux de données prétraités
# ---------------------------------------------------------------------

from pathlib import Path


# ---------------------------------------------------------------------
# 1. Détermination de la racine du projet
# ---------------------------------------------------------------------

project_root = Path.cwd().resolve()

while (
    project_root.parent != project_root
    and not (project_root / "pyproject.toml").exists()
):
    project_root = project_root.parent

if not (project_root / "pyproject.toml").exists():
    raise FileNotFoundError(
        "Impossible de déterminer la racine du projet."
    )


# ---------------------------------------------------------------------
# 2. Répertoire de sauvegarde
# ---------------------------------------------------------------------

processed_data_dir = (
    project_root
    / "data"
    / "processed"
)

processed_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# 3. Fichiers de sortie
# ---------------------------------------------------------------------

x_train_path = (
    processed_data_dir
    / "X_train_processed.parquet"
)

x_test_path = (
    processed_data_dir
    / "X_test_processed.parquet"
)

y_train_path = (
    processed_data_dir
    / "y_train.parquet"
)

y_test_path = (
    processed_data_dir
    / "y_test.parquet"
)


# ---------------------------------------------------------------------
# 4. Sauvegarde
# ---------------------------------------------------------------------

X_train_processed.to_parquet(
    x_train_path,
    index=True,
)

X_test_processed.to_parquet(
    x_test_path,
    index=True,
)

y_train.to_frame().to_parquet(
    y_train_path,
    index=True,
)

y_test.to_frame().to_parquet(
    y_test_path,
    index=True,
)


# ---------------------------------------------------------------------
# 5. Vérification de la sauvegarde
# ---------------------------------------------------------------------

saved_files = {
    "X_train_processed": x_train_path,
    "X_test_processed": x_test_path,
    "y_train": y_train_path,
    "y_test": y_test_path,
}

missing_saved_files = [
    name
    for name, path in saved_files.items()
    if not path.exists()
]

if missing_saved_files:
    raise FileNotFoundError(
        "Échec de sauvegarde pour : "
        + ", ".join(missing_saved_files)
    )


# ---------------------------------------------------------------------
# 6. Rapport métier de sauvegarde
# ---------------------------------------------------------------------

saved_datasets_report_df = pd.DataFrame(
    [
        {
            "dataset": "X_train_processed",
            "role": "Variables explicatives - entraînement",
            "observations": X_train_processed.shape[0],
            "variables": X_train_processed.shape[1],
            "statut": "Sauvegardé",
        },
        {
            "dataset": "X_test_processed",
            "role": "Variables explicatives - test",
            "observations": X_test_processed.shape[0],
            "variables": X_test_processed.shape[1],
            "statut": "Sauvegardé",
        },
        {
            "dataset": "y_train",
            "role": "Variable cible - entraînement",
            "observations": len(y_train),
            "variables": 1,
            "statut": "Sauvegardé",
        },
        {
            "dataset": "y_test",
            "role": "Variable cible - test",
            "observations": len(y_test),
            "variables": 1,
            "statut": "Sauvegardé",
        },
    ]
)

display(saved_datasets_report_df)


print(
    "\n✅ Les quatre jeux de données prétraités "
    "ont été sauvegardés dans data/processed/."
)

,dataset,role,observations,variables,statut
0,X_train_processed,Variables explicatives - entraînement,80000,32,Sauvegardé
1,X_test_processed,Variables explicatives - test,20000,32,Sauvegardé
2,y_train,Variable cible - entraînement,80000,1,Sauvegardé
3,y_test,Variable cible - test,20000,1,Sauvegardé



✅ Les quatre jeux de données prétraités ont été sauvegardés dans data/processed/.


### 10.19 Sauvegarde des artefacts de preprocessing

Les jeux de données prétraités ont été sauvegardés au point précédent.

Afin de rendre les transformations reproductibles sur de nouvelles données,
les objets et paramètres appris exclusivement à partir des données
d'entraînement doivent également être persistés.

Les artefacts sauvegardés sont :

- la table de fréquences utilisée pour le Frequency Encoding de
  `manufacturer_make` ;
- le `OneHotEncoder` appris sur les variables catégorielles concernées ;
- le `StandardScaler` appris sur les variables numériques continues ;
- les métadonnées décrivant la structure finale du preprocessing.

Les métadonnées permettent notamment de conserver :

- les variables traitées par One-Hot Encoding ;
- les variables standardisées ;
- les indicateurs binaires créés ;
- les noms des variables générées par One-Hot Encoding ;
- l'ordre exact des variables finales utilisées pour la modélisation.

Ces artefacts sont enregistrés dans le répertoire `models/preprocessing/`.

Ils permettront d'appliquer ultérieurement aux nouvelles données les mêmes
transformations que celles apprises sur Train, sans recalculer les paramètres
à partir des données d'inférence.

In [29]:
# ---------------------------------------------------------------------
# 10.19 - Sauvegarde des artefacts de preprocessing
# ---------------------------------------------------------------------

import json
import joblib


# ---------------------------------------------------------------------
# 1. Vérification de la racine du projet
# ---------------------------------------------------------------------

if "project_root" not in globals():
    project_root = Path.cwd().resolve()

    while (
        project_root.parent != project_root
        and not (project_root / "pyproject.toml").exists()
    ):
        project_root = project_root.parent

    if not (project_root / "pyproject.toml").exists():
        raise FileNotFoundError(
            "Impossible de déterminer la racine du projet."
        )


# ---------------------------------------------------------------------
# 2. Répertoire de sauvegarde des artefacts
# ---------------------------------------------------------------------

preprocessing_dir = (
    project_root
    / "models"
    / "preprocessing"
)

preprocessing_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# 3. Définition des fichiers de sortie
# ---------------------------------------------------------------------

frequency_map_path = (
    preprocessing_dir
    / "manufacturer_frequency_map.joblib"
)

onehot_encoder_path = (
    preprocessing_dir
    / "onehot_encoder.joblib"
)

standard_scaler_path = (
    preprocessing_dir
    / "standard_scaler.joblib"
)

metadata_path = (
    preprocessing_dir
    / "preprocessing_metadata.json"
)


# ---------------------------------------------------------------------
# 4. Sauvegarde des objets appris sur Train
# ---------------------------------------------------------------------

joblib.dump(
    manufacturer_frequency_map,
    frequency_map_path,
)

joblib.dump(
    onehot_encoder,
    onehot_encoder_path,
)

joblib.dump(
    standard_scaler,
    standard_scaler_path,
)


# ---------------------------------------------------------------------
# 5. Construction des métadonnées du preprocessing
# ---------------------------------------------------------------------

preprocessing_metadata = {

    "frequency_encoding": {
        "source_column": "manufacturer_make",
        "output_column": "manufacturer_make_frequency",
        "unknown_value": 0.0,
    },

    "onehot_encoding": {
        "source_columns": list(onehot_columns),
        "output_columns": list(onehot_feature_names),
        "handle_unknown": "ignore",
    },

    "binary_indicators": list(
        binary_indicator_columns
    ),

    "standardization": {
        "columns": list(columns_to_scale),
    },

    "final_features": (
        X_train_processed
        .columns
        .tolist()
    ),
}


# ---------------------------------------------------------------------
# 6. Sauvegarde des métadonnées
# ---------------------------------------------------------------------

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        preprocessing_metadata,
        file,
        indent=4,
        ensure_ascii=False,
    )


# ---------------------------------------------------------------------
# 7. Vérification de la sauvegarde
# ---------------------------------------------------------------------

saved_artifacts = {
    "Frequency Encoding": frequency_map_path,
    "OneHotEncoder": onehot_encoder_path,
    "StandardScaler": standard_scaler_path,
    "Métadonnées preprocessing": metadata_path,
}

missing_artifacts = [
    name
    for name, path in saved_artifacts.items()
    if not path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "Échec de sauvegarde pour : "
        + ", ".join(missing_artifacts)
    )


# ---------------------------------------------------------------------
# 8. Rapport fonctionnel
# ---------------------------------------------------------------------

artifacts_report_df = pd.DataFrame(
    [
        {
            "artefact": "Frequency Encoding",
            "role": "Encodage de manufacturer_make",
            "statut": "Sauvegardé",
        },
        {
            "artefact": "OneHotEncoder",
            "role": "Encodage des variables catégorielles",
            "statut": "Sauvegardé",
        },
        {
            "artefact": "StandardScaler",
            "role": "Standardisation des variables numériques",
            "statut": "Sauvegardé",
        },
        {
            "artefact": "Métadonnées preprocessing",
            "role": "Structure et paramètres du preprocessing",
            "statut": "Sauvegardé",
        },
    ]
)

display(artifacts_report_df)


print(
    "\n✅ Les artefacts de preprocessing "
    "ont été sauvegardés dans models/preprocessing/."
)

,artefact,role,statut
0,Frequency Encoding,Encodage de manufacturer_make,Sauvegardé
1,OneHotEncoder,Encodage des variables catégorielles,Sauvegardé
2,StandardScaler,Standardisation des variables numériques,Sauvegardé
3,Métadonnées preprocessing,Structure et paramètres du preprocessing,Sauvegardé



✅ Les artefacts de preprocessing ont été sauvegardés dans models/preprocessing/.


### 10.20 Validation du rechargement des artefacts de preprocessing

Les artefacts de preprocessing ayant été sauvegardés, leur rechargement est
contrôlé avant de clôturer le notebook.

Cette étape vérifie que :

- la table de fréquences de `manufacturer_make` peut être rechargée ;
- le `OneHotEncoder` peut être rechargé ;
- le `StandardScaler` peut être rechargé ;
- les métadonnées JSON peuvent être relues ;
- le nombre et l'ordre des variables finales enregistrées dans les métadonnées
  correspondent à la structure de `X_train_processed` et
  `X_test_processed`.

Aucun nouvel apprentissage ni aucune transformation des données n'est effectué
à cette étape.

L'objectif est de vérifier que les artefacts persistés sont effectivement
réutilisables pour les prochaines étapes du pipeline Machine Learning et pour
le futur traitement de nouvelles données.

In [30]:
# ---------------------------------------------------------------------
# 10.20 - Validation du rechargement des artefacts de preprocessing
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 1. Rechargement des artefacts sauvegardés
# ---------------------------------------------------------------------

loaded_frequency_map = joblib.load(
    frequency_map_path
)

loaded_onehot_encoder = joblib.load(
    onehot_encoder_path
)

loaded_standard_scaler = joblib.load(
    standard_scaler_path
)

with open(
    metadata_path,
    "r",
    encoding="utf-8",
) as file:
    loaded_metadata = json.load(file)


# ---------------------------------------------------------------------
# 2. Récupération des informations enregistrées
# ---------------------------------------------------------------------

saved_final_features = loaded_metadata[
    "final_features"
]

saved_onehot_columns = loaded_metadata[
    "onehot_encoding"
]["source_columns"]

saved_onehot_output_columns = loaded_metadata[
    "onehot_encoding"
]["output_columns"]

saved_scaling_columns = loaded_metadata[
    "standardization"
]["columns"]

saved_binary_indicators = loaded_metadata[
    "binary_indicators"
]

frequency_source_column = loaded_metadata[
    "frequency_encoding"
]["source_column"]

frequency_output_column = loaded_metadata[
    "frequency_encoding"
]["output_column"]


# ---------------------------------------------------------------------
# 3. Structure actuelle de Train et Test
# ---------------------------------------------------------------------

train_final_features = (
    X_train_processed
    .columns
    .tolist()
)

test_final_features = (
    X_test_processed
    .columns
    .tolist()
)


metadata_matches_train = (
    saved_final_features
    == train_final_features
)

metadata_matches_test = (
    saved_final_features
    == test_final_features
)

train_matches_test = (
    train_final_features
    == test_final_features
)


# ---------------------------------------------------------------------
# 4. Validation des objets rechargés
# ---------------------------------------------------------------------

frequency_map_valid = (
    len(loaded_frequency_map) > 0
)

onehot_encoder_valid = (
    hasattr(
        loaded_onehot_encoder,
        "categories_",
    )
)

standard_scaler_valid = (
    hasattr(
        loaded_standard_scaler,
        "mean_",
    )
    and hasattr(
        loaded_standard_scaler,
        "scale_",
    )
)


# ---------------------------------------------------------------------
# 5. Rapport général de validation
# ---------------------------------------------------------------------

artifact_validation_df = pd.DataFrame(
    [
        {
            "artefact": "Frequency Encoding",
            "role": (
                f"{frequency_source_column} "
                f"→ {frequency_output_column}"
            ),
            "verification": (
                "Table de fréquences rechargeable"
            ),
            "statut": (
                "OK"
                if frequency_map_valid
                else "À vérifier"
            ),
        },
        {
            "artefact": "OneHotEncoder",
            "role": (
                "Encodage des variables catégorielles "
                "à faible cardinalité"
            ),
            "verification": (
                "Encodeur et catégories apprises rechargeables"
            ),
            "statut": (
                "OK"
                if onehot_encoder_valid
                else "À vérifier"
            ),
        },
        {
            "artefact": "StandardScaler",
            "role": (
                "Standardisation des variables "
                "numériques continues"
            ),
            "verification": (
                "Moyennes et écarts-types appris rechargeables"
            ),
            "statut": (
                "OK"
                if standard_scaler_valid
                else "À vérifier"
            ),
        },
        {
            "artefact": "Métadonnées",
            "role": (
                "Structure finale des variables"
            ),
            "verification": (
                "Métadonnées identiques à X_train_processed"
            ),
            "statut": (
                "OK"
                if metadata_matches_train
                else "À vérifier"
            ),
        },
        {
            "artefact": "Métadonnées",
            "role": (
                "Structure finale des variables"
            ),
            "verification": (
                "Métadonnées identiques à X_test_processed"
            ),
            "statut": (
                "OK"
                if metadata_matches_test
                else "À vérifier"
            ),
        },
        {
            "artefact": "Train / Test",
            "role": (
                "Compatibilité pour la modélisation"
            ),
            "verification": (
                "Même nombre, mêmes noms et même ordre "
                "des variables"
            ),
            "statut": (
                "OK"
                if train_matches_test
                else "À vérifier"
            ),
        },
    ]
)

print("VALIDATION GÉNÉRALE DES ARTEFACTS\n")

display(
    artifact_validation_df
)


# ---------------------------------------------------------------------
# 6. Détail du Frequency Encoding
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("1. FREQUENCY ENCODING")
print("=" * 70)

print(
    f"\nVariable source : "
    f"{frequency_source_column}"
)

print(
    f"Variable créée : "
    f"{frequency_output_column}"
)

print(
    f"Nombre de modalités apprises sur Train : "
    f"{len(loaded_frequency_map)}"
)

print(
    "\nRôle : les fréquences apprises sur Train "
    "pourront être réutilisées sur de nouvelles données."
)


# ---------------------------------------------------------------------
# 7. Détail du One-Hot Encoding
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("2. ONE-HOT ENCODING")
print("=" * 70)

onehot_details = []

for column, categories in zip(
    saved_onehot_columns,
    loaded_onehot_encoder.categories_,
):

    onehot_details.append(
        {
            "variable_source": column,
            "nombre_modalites_apprises": len(categories),
            "modalites_apprises": ", ".join(
                map(str, categories)
            ),
        }
    )


onehot_details_df = pd.DataFrame(
    onehot_details
)

display(
    onehot_details_df
)

print(
    f"\nNombre total de variables binaires créées "
    f"par One-Hot Encoding : "
    f"{len(saved_onehot_output_columns)}"
)


# ---------------------------------------------------------------------
# 8. Détail du StandardScaler
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("3. STANDARDISATION")
print("=" * 70)

scaler_details_df = pd.DataFrame(
    {
        "variable": saved_scaling_columns,
        "moyenne_apprise_train": (
            loaded_standard_scaler.mean_
        ),
        "ecart_type_appris_train": (
            loaded_standard_scaler.scale_
        ),
    }
)

display(
    scaler_details_df
)

print(
    f"\nNombre de variables standardisées : "
    f"{len(saved_scaling_columns)}"
)


# ---------------------------------------------------------------------
# 9. Détail des indicateurs binaires
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("4. INDICATEURS BINAIRES CONSERVÉS")
print("=" * 70)

binary_indicators_df = pd.DataFrame(
    {
        "indicateur": saved_binary_indicators,
        "traitement": (
            ["Conservé en 0/1 sans standardisation"]
            * len(saved_binary_indicators)
        ),
    }
)

display(
    binary_indicators_df
)


# ---------------------------------------------------------------------
# 10. Structure finale utilisée pour la modélisation
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("5. STRUCTURE FINALE")
print("=" * 70)

print(
    f"\nNombre total de variables finales : "
    f"{len(saved_final_features)}"
)

final_features_df = pd.DataFrame(
    {
        "position": range(
            1,
            len(saved_final_features) + 1,
        ),
        "variable_finale": saved_final_features,
    }
)

display(
    final_features_df
)


# ---------------------------------------------------------------------
# 11. Validation stricte
# ---------------------------------------------------------------------

all_artifacts_valid = all(
    [
        frequency_map_valid,
        onehot_encoder_valid,
        standard_scaler_valid,
        metadata_matches_train,
        metadata_matches_test,
        train_matches_test,
    ]
)

if not all_artifacts_valid:

    raise ValueError(
        "La validation des artefacts "
        "de preprocessing a échoué."
    )


print(
    "\n✅ Validation réussie : les artefacts sauvegardés "
    "sont rechargeables et correspondent à la structure "
    "finale utilisée pour la modélisation."
)

VALIDATION GÉNÉRALE DES ARTEFACTS



,artefact,role,verification,statut
0,Frequency Encoding,manufacturer_make → manufacturer_make_frequency,Table de fréquences rechargeable,OK
1,OneHotEncoder,Encodage des variables catégorielles à faible ...,Encodeur et catégories apprises rechargeables,OK
2,StandardScaler,Standardisation des variables numériques conti...,Moyennes et écarts-types appris rechargeables,OK
3,Métadonnées,Structure finale des variables,Métadonnées identiques à X_train_processed,OK
4,Métadonnées,Structure finale des variables,Métadonnées identiques à X_test_processed,OK
5,Train / Test,Compatibilité pour la modélisation,"Même nombre, mêmes noms et même ordre des vari...",OK



1. FREQUENCY ENCODING

Variable source : manufacturer_make
Variable créée : manufacturer_make_frequency
Nombre de modalités apprises sur Train : 93

Rôle : les fréquences apprises sur Train pourront être réutilisées sur de nouvelles données.

2. ONE-HOT ENCODING


,variable_source,nombre_modalites_apprises,modalites_apprises
0,vehicle_category_type,2,"M1, N1"
1,fuel_type,9,"diesel, diesel/electric, e85, electric, hydrog..."
2,fuel_mode,6,"B, E, F, H, M, P"



Nombre total de variables binaires créées par One-Hot Encoding : 17

3. STANDARDISATION


...


Nombre de variables standardisées : 11

4. INDICATEURS BINAIRES CONSERVÉS


,indicateur,traitement
0,has_electric_energy_consumption_wh_km,Conservé en 0/1 sans standardisation
1,has_electric_range_km,Conservé en 0/1 sans standardisation
2,has_fuel_consumption,Conservé en 0/1 sans standardisation
3,has_co2_reduction_wltp_g_km,Conservé en 0/1 sans standardisation



5. STRUCTURE FINALE

Nombre total de variables finales : 32


,position,variable_finale
0,1,mass_running_order_kg
1,2,wltp_test_mass_kg
2,3,engine_capacity_cm3
3,4,engine_power_kw
4,5,electric_energy_consumption_wh_km
5,6,co2_reduction_wltp_g_km
6,7,fuel_consumption
7,8,electric_range_km
8,9,registration_month_sin
9,10,registration_month_cos



✅ Validation réussie : les artefacts sauvegardés sont rechargeables et correspondent à la structure finale utilisée pour la modélisation.


### 10.21 Conclusion du preprocessing Train / Test

Le preprocessing des données d'entraînement et de test est désormais terminé,
sauvegardé et validé.

Les traitements réalisés dans ce notebook ont permis de construire des jeux de
données directement exploitables pour la phase de modélisation, tout en
respectant la séparation entre Train et Test afin d'éviter les fuites
d'information.

Les principales étapes réalisées sont :

- séparation des variables explicatives et de la variable cible ;
- constitution de `X_train`, `X_test`, `y_train` et `y_test` ;
- analyse des valeurs manquantes et distinction entre absences structurelles
  et résiduelles ;
- application des règles métier retenues pour les valeurs manquantes
  structurelles ;
- imputation des valeurs manquantes résiduelles à partir de statistiques
  apprises exclusivement sur Train ;
- création des indicateurs binaires permettant de conserver l'information
  associée à certaines absences de données ;
- traitement des valeurs manquantes catégorielles ;
- Frequency Encoding de `manufacturer_make`, appris exclusivement sur Train ;
- One-Hot Encoding de `vehicle_category_type`, `fuel_type` et `fuel_mode`,
  appris exclusivement sur Train ;
- standardisation des variables numériques continues à partir des paramètres
  appris exclusivement sur Train ;
- conservation sans standardisation des variables binaires ;
- validation de l'absence de valeurs manquantes, de valeurs infinies et de
  variables non numériques ;
- vérification de la cohérence des structures finales entre Train et Test ;
- sauvegarde des jeux de données prétraités au format Parquet ;
- sauvegarde des artefacts de preprocessing et des métadonnées associées ;
- rechargement et validation des artefacts sauvegardés.

Les matrices finales `X_train_processed` et `X_test_processed` possèdent la
même structure de variables et sont cohérentes avec leurs cibles respectives.

Les artefacts nécessaires à la reproduction du preprocessing ont également
été persistés :

- table de fréquences de `manufacturer_make` ;
- `OneHotEncoder` ;
- `StandardScaler` ;
- métadonnées décrivant les transformations et l'ordre final des variables.

Le preprocessing effectué dans ce notebook est ainsi reproductible sur de
nouvelles données sans réapprendre les paramètres à partir des données
d'inférence.

Les données prétraitées sont maintenant prêtes pour l'étape suivante du projet :
**l'entraînement, la comparaison et l'évaluation des modèles de Machine
Learning**.